# Meenakshi Polymers – generic OCR pipeline v3 (Qwen3-VL, Colab GPU)

Feed any scanned PDF → every page is read, matched to a **form spec** (`forms/*.json`) and returned as one JSON
per page in which **every cell carries a confidence score**. Review files highlight the cells a person should check.

```
PDF page
  ├─ A. Prepare page         render, deskew, flatten lighting, ruled lines
  ├─ B. Read header          title lines, doc no, date / shift / month, every "label: value" pair
  ├─ C. Read table skeleton  column labels, a few row labels, printed rows or handwritten entries
  ├─ D. Match form           score the page against every forms/*.json spec
  │        ├─ matched  → the spec gives keys, types, cell format, rules
  │        └─ no match → generic keys from the printed labels, draft spec in forms/_drafts, status new_form
  ├─ E. Read body            record (one entry per call) · matrix (one printed row per call) · block_grid (part × day)
  ├─ F. Confidence           token probabilities of every value (no self-rating by the model)
  ├─ G. Normalise + check    numbers, dates, dash, ditto, tally counts; row / section / summary totals; spec rules
  │                          failed rows: re-read with a tight crop, or swap in the model's 2nd-choice digit
  └─ H. Write                json/ · review.xlsx (coloured cells) · review.html (editable, coloured) · csv/
```

| Layout kind | Forms |
|---|---|
| `record` | supplier lot rejection / rejection / rework / segregation, line rejection (press, weld, penetration), scrap note |
| `matrix` | MIS FI visual / plating / powder coating QA / powder coating daily, QG-2 weld checksheets (tally counts), weld shop rework & rejection sheet (2 shift blocks) |
| `block_grid` | BS6 straight pass monitoring (weld1, weld2, monthly handwritten) |

**Confidence.** Generation records the probability of every token. A value's `model` score is the *lowest*
probability among its tokens; other tokens with p ≥ 0.05 become `alternatives`. Checks then adjust it: a passing
total lifts it to ≥ 0.95, a failing one caps it at 0.50, two disagreeing reads cap it at 0.40.
Status: `ok` ≥ 0.90 · `review` ≥ 0.60 · `low` < 0.60 · `repaired` · `no_count` (tally without a written count) · `empty`.

**Adding a form:** run the PDF once → `forms/_drafts/<id>.json` → fix keys / types / rules → move it to `forms/` → re-run.

Accuracy validation against ground-truth CSVs is not part of this notebook (see v2).

## 1. Install

In [ ]:
import os, sys, subprocess
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation on small GPUs
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Qwen3-VL needs transformers >= 4.57
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "transformers>=4.57.0", "accelerate", "pymupdf", "opencv-python-headless", "openpyxl"],
                   check=True)
print("Running in Colab:", IN_COLAB)

In [ ]:
import re, ast, csv, json, time, shutil, html, fnmatch, difflib, traceback
from pathlib import Path
import numpy as np
import cv2
import pandas as pd
import pymupdf
import matplotlib
import matplotlib.pyplot as plt

try:
    from IPython.display import display, HTML
except ImportError:                       # plain python (local testing)
    display, HTML = print, str

try:
    import torch
    HAS_CUDA = torch.cuda.is_available()
    print("torch", torch.__version__, "| CUDA:", HAS_CUDA)
    if HAS_CUDA:
        p = torch.cuda.get_device_properties(0)
        print(f"GPU: {p.name}  {p.total_memory / 1e9:.1f} GB")
except ImportError:
    torch, HAS_CUDA = None, False
    print("torch not installed")
print("opencv", cv2.__version__, "| numpy", np.__version__, "| pymupdf", pymupdf.__version__)

## 2. Configuration

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/MeenakshiPublic"

def mount_drive():
    if not IN_COLAB:
        return
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")

mount_drive()

# files or folders (searched recursively)
INPUT_PATHS = ([f"{DRIVE_ROOT}/Meenakshi ALL Documents"] if IN_COLAB
               else [os.environ.get("MP_INPUT", "Dataset/Meenakshi ALL Documents")])
OUTPUT_DIR  = Path(f"{DRIVE_ROOT}/ocr_v3_output") if IN_COLAB else Path(os.environ.get("MP_OUTPUT", "ocr_v3_output"))
# form specs: one JSON per form. Missing built-in specs are written here on the first run, existing files are never
# overwritten, so edits made in this folder win. New forms get a draft in FORMS_DIR/_drafts.
FORMS_DIR   = Path(f"{DRIVE_ROOT}/forms") if IN_COLAB else Path(os.environ.get("MP_FORMS", "forms"))
WRITE_DRAFTS = True
ONLY_FILES  = []          # substrings of the path to process only some files, e.g. ["SCRAP", "QG", "REWORK"]
MAX_PAGES_PER_PDF = None  # e.g. 1 for a quick look

# ---- form choice ---------------------------------------------------------------------------------
# FORM_OVERRIDE: glob on the file name or on the path relative to the input folder -> form_id. Always wins.
FORM_OVERRIDE = {}        # e.g. {"scan_0412.pdf": "scrap_note", "*QG*": "weld_qg2_checksheet"}
# FOLDER_FORMS: folder (relative to the input folder) -> form_id, the deepest matching folder wins. Used when the page
# itself gives no clear match (or always, with TRUST_FOLDER = True). Mixed folders are simply not listed.
FOLDER_FORMS = {
    "line_rejection_report":        "line_rejection",
    "scrap_note":                   "scrap_note",
    "supplier_lot_rejection":       "supplier_lot_rejection_summary",
    "supplier_rejection":           "supplier_rejection",
    "supplier_rework":              "supplier_rework",
    "supplier_segregation_report":  "supplier_segregation",
    "weld_data":                    "st_pass_monitoring",
    "mis_report/mis_type_1":        "mis_fi_visual_pc",
    "mis_report/mis_type_2":        "mis_plating",
    "mis_report/mis_type_3":        "mis_pc_daily_qa",
    "mis_report/mis_type_4":        "mis_pc_daily",
    # "Fwd_ Fwd_ IOT WELD SHOP" holds several forms (QG-2, rework sheet, straight pass, penetration): matched by page
}
TRUST_FOLDER = False      # True = the folder decides whenever it maps to a form
MATCH_MIN    = 0.60       # best score needed for a match
MATCH_MARGIN = 0.08       # ... and this far ahead of the runner-up (forms of the same family, e.g. MIS, are exempt)
FOLDER_MIN   = 0.30       # the folder's form is used as fallback only if the page scores at least this for it
if not IN_COLAB and os.environ.get("MP_OVERRIDE"):
    FORM_OVERRIDE = json.loads(os.environ["MP_OVERRIDE"])
if not IN_COLAB and os.environ.get("MP_ONLY"):
    ONLY_FILES = os.environ["MP_ONLY"].split(",")

# ---- confidence ----------------------------------------------------------------------------------
CONF_OK, CONF_REVIEW = 0.90, 0.60     # final >= CONF_OK -> ok, >= CONF_REVIEW -> review, below -> low
CHECK_PASS_FLOOR = 0.95               # a passing total lifts the cells it covers to at least this
CHECK_FAIL_CAP   = 0.50               # a failing total caps them at this
DISAGREE_CAP     = 0.40               # two reads of the same cell that disagree
REPAIRED_CAP     = 0.85               # a value changed by a repair is never better than "review"
ALT_MIN_P        = 0.05               # keep 2nd-choice tokens at least this likely as alternatives
TOP_K            = 5                  # tokens recorded per generation step
MAX_REREAD_ROWS  = 12                 # matrix rows re-read (tight crop) per page when their total does not add up

# ---- model ---------------------------------------------------------------------------------------
MODEL_ID       = "Qwen/Qwen3-VL-4B-Instruct"   # "Qwen/Qwen3-VL-8B-Instruct" on L4/A100 for better handwriting
DRY_RUN        = False    # True = preprocessing + saved model inputs only, no model
MAX_NEW_TOKENS = 2048
# Pixel cap for every image sent to the model (memory grows with the SQUARE of the pixel count).
# None = automatic: 1.0 MP on GPUs < 20 GB (T4), 2.0 MP on L4 / A100.
MAX_IMAGE_PIXELS = None

# ---- debugging -----------------------------------------------------------------------------------
VERBOSE         = True    # one line per model call (+ start of the answer)
SHOW_INLINE     = True    # show overlays and crops in the notebook while running
SHOW_MAX_IMAGES = 4       # inline crops per page
SAVE_MODEL_VIEW = True    # also save every input at the resolution Qwen processed it
INCLUDE_META_IN_JSON = True   # "_meta" (form decision, issues, debug paths) in the JSON files

# ---- preprocessing (pixel values at 2480 px page width ≈ A4 at 300 dpi) -------------------------
PORTRAIT_WIDTH, LANDSCAPE_WIDTH = 2480, 3508
HEADER_BAND_FRAC = {"portrait": 0.24, "landscape": 0.17}   # top part of the page read as the header
MIN_ROW_INK          = 1500   # record forms: handwriting pixels for a ruled row to count as filled
CONTINUATION_MAX_INK = 150    # record forms: less ink in the first column -> row continues the previous entry
TOP_PAD, BOTTOM_PAD, SIDE_PAD, IMAGE_MARGIN = 10, 22, 40, 20
OVERFLOW_WORD_GAP  = (25, 9)  # px (x, y) gap that still joins strokes into one written word
OVERFLOW_MAX_ROWS  = 0.9      # a word may reach at most this many row heights above / below its entry
OVERFLOW_MAX_SIDE  = 450      # px a word may run past the left / right table border
MIS_MIN_ROWS      = 15        # matrix: fewer printed rows found than this -> page fails (layout needs review)
MIS_ROW_REACH     = 0.5       # matrix: handwriting may stick out this many row heights above / below its row
ROW_BATCH         = 4         # matrix rows sent to the GPU together (1 = one at a time; lower it on out-of-memory)
ST_MIN_DAY_INK    = 400       # block grid: ink needed in a part×day cell group to send it to the model

if not DRY_RUN and not HAS_CUDA:
    print("⚠ No GPU found -> DRY_RUN (preprocessing only). Colab: Runtime → Change runtime type → T4 GPU")
    DRY_RUN = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("input :", INPUT_PATHS)
print("output:", OUTPUT_DIR.resolve())
print("forms :", FORMS_DIR.resolve())
print("DRY_RUN:", DRY_RUN, "| override:", FORM_OVERRIDE or "-", "| trust folder:", TRUST_FOLDER)

### PDFs to process

In [ ]:
def list_pdfs(paths):
    found = []
    for p in map(Path, paths):
        if p.is_file() and p.suffix.lower() == ".pdf":
            found.append((p, p.name))
        elif p.is_dir():
            for f in sorted(p.rglob("*")):
                if f.is_file() and f.suffix.lower() == ".pdf":
                    found.append((f, f.relative_to(p).as_posix()))
        else:
            print("⚠ not found:", p)
    if ONLY_FILES:
        found = [(f, r) for f, r in found if any(s.upper() in r.upper() for s in ONLY_FILES)]
    return found

PDFS = list_pdfs(INPUT_PATHS)
print(f"{len(PDFS)} PDF(s)")
for f, rel in PDFS:
    print("  ", rel)

## 3. Form specs (`forms/*.json`)
A spec is data only:
- `match` – `title_any` / `title_none` phrases, `doc_no_any` fragments, `columns_any` printed column labels
- `header` – output key → printed labels of that header field
- `layout.kind` – `record` | `matrix` | `block_grid`, plus kind options (below)
- `columns` – `key`, printed `labels`, `type` (`int`, `number`, `text`, `date`, `qty_text`), and for matrices a `role`
  (`label` printed row name · `text` handwritten text · `part` summed into `total` · `total` · `disposition` summed into
  `grand_total` · `grand_total` · `value` · `ignore`), optional `group` (two-level header), `cell_format`
  (`plain` | `tally_with_count`), `required`, `pattern` (regex)
- `rules` – expressions on one record / row (`lot_qty == rejection_qty + ok_qty`); `summary_rows` + `summary_rules`
  – expressions on the summary rows of a matrix, evaluated per value column

Kind options – record: `header_rows` (ruled lines above the data; auto when missing), `footer_rows`,
`expected_col_lines`. Matrix: `row_id` (`sr_no` | `label`), `row_name_key`, `row_strip_cols` (column-line indexes of
the printed row-name strip), `header_terms`, `require_total`, `min_rows`, `allow_gaps`, `blocks`, `fixed_rows`,
`footer_terms`, `row_sum_on_summary`. Block grid: `params`, `shifts`, `days_per_sheet`, `header_bands`.

In [ ]:
def col(key, *labels, type="int", role=None, group=None, fmt=None, required=False, pattern=None):
    c = dict(key=key, labels=list(labels) or [key.upper()], type=type)
    for k, v in (("role", role), ("group", group), ("cell_format", fmt), ("pattern", pattern)):
        if v is not None:
            c[k] = v
    if required:
        c["required"] = True
    return c

def T(key, *labels, **kw): return col(key, *labels, type="text", **kw)
def D(key, *labels, **kw): return col(key, *labels, type="date", **kw)
def L(key, *labels):       return col(key, *labels, type="text", role="label")
def P(key, *labels, **kw): return col(key, *labels, role="part", **kw)

RECORD_HEADER = {"company": ["COMPANY"], "title": ["TITLE"], "month": ["MONTH"]}
SUPPLIER_COLS_START = [D("date", "DATE", required=True), T("part_name", "PART NAME", required=True),
                       T("supplier_name", "SUPPLIER NAME")]

PC_PARTS = [P("sg_aah", "SG-AAH"), P("sg_aae", "SG-AAE"), P("sw_aah", "SW (AAH)"), P("sp_aae", "SP-AAE"),
            P("sg_aacy", "SG (AACY)"), P("sw_aacy", "SW (AACY)"), P("eg_kvh", "EG-KVH"),
            P("cross_pipe_aael", "CROSS PIPE-AAEL"), P("handle_abza", "HANDLE ABZA"), P("eg_abza", "EG-ABZA"),
            P("pb_abz", "PB-ABZ"), P("pgc_ack", "PGC-ACK"), P("pb_ack", "PB-ACK"), P("handle_aack", "HANDLE AACK"),
            P("side_stand", "SIDE STAND"), P("sg_aael", "SG-AAEL")]
MIS_LABELS = [L("sr_no", "SR.NO", "S.NO"), L("defect", "DEFECT", "DESCRIPTION OF DEFECT"), L("reason", "REASON")]
MIS_HEADER = {"company": ["COMPANY"], "title": ["TITLE"], "doc_no": ["DOC NO"], "rev_no": ["REV NO"],
              "rev_date": ["REV DATE"], "org_date": ["ORG DATE"], "report": ["DAILY MIS REPORT"],
              "shift": ["SHIFT"], "date": ["DATE"], "month": ["MONTH"]}
MIS_LAYOUT = dict(kind="matrix", row_id="sr_no", row_name_key="defect", min_rows=15)
def total(key="total", *labels): return col(key, *(labels or ("TOTAL",)), role="total")
def disp(key, *labels, **kw):     return col(key, *labels, role="disposition", **kw)

REWORK_PARTS = ["Carrier AAD Manual", "Carrier AAD Robot", "Carrier AAEL Robot", "Engine Guard KST", "SARI GUARD AAH",
                "SARI GUARD AAE", "SARI GUARD ACY", "STEP PILLION AAE/KCC", "STEP WOMEN AAH", "PGC AAE", "PGC ACKA",
                "BRAKE PADEL AAEN", "Main Stand AAE", "Main Stand ACKA", "PEDAL BRAKE ABZA DD", "PEDAL BRAKE ABZA SD",
                "Cross Pipe AAD", "Cross Pipe AAEL", "SIDE STAND AAF", "SIDE STAND ABZA", "ENGINE GUARD ABZA"]

BUILTIN_FORMS = [
    # ---------------------------------------------------------------- record forms
    dict(form_id="supplier_lot_rejection_summary", version=1, name="Supplier Lot Rejection Summary", family="supplier",
         match=dict(title_any=["SUPPLIER LOT REJECTION SUMMARY", "LOT REJECTION SUMMARY"],
                    columns_any=["DATE", "PART NAME", "SUPPLIER NAME", "LOT QTY", "REW QTY", "REJ QTY", "OK QTY",
                                 "PROBLEM DESCRIPTION", "ACTION", "RETURN STATUS", "CAPA STATUS", "REMARKS"]),
         header=RECORD_HEADER,
         layout=dict(kind="record", header_rows=3, footer_rows=1, expected_col_lines=13),
         columns=SUPPLIER_COLS_START + [col("lot_qty", "LOT QTY", required=True), col("rework_qty", "REW QTY"),
                  col("rejection_qty", "REJ QTY"), col("ok_qty", "OK QTY", required=True),
                  T("problem_description", "PROBLEM DESCRIPTION"), T("action", "ACTION"),
                  T("return_status", "RETURN STATUS"), T("capa_status", "CAPA STATUS"), T("remarks", "REMARKS")],
         rules=["lot_qty == rejection_qty + ok_qty"]),
    dict(form_id="supplier_segregation", version=1, name="Supplier Segregation Report", family="supplier",
         match=dict(title_any=["SUPPLIER SEGREGATION REPORT", "SEGREGATION REPORT"],
                    columns_any=["DATE", "PART NAME", "SUPPLIER NAME", "LOT QTY", "SEGREGATION QTY", "OK QTY",
                                 "PROBLEM DESCRIPTION", "REMARKS"]),
         header=RECORD_HEADER,
         layout=dict(kind="record", header_rows=3, footer_rows=1, expected_col_lines=9),
         columns=SUPPLIER_COLS_START + [col("lot_qty", "LOT QTY", required=True),
                  col("segregation_qty", "SEGREGATION QTY", required=True), col("ok_qty", "OK QTY", required=True),
                  T("problem_description", "PROBLEM DESCRIPTION"), T("remarks", "REMARKS")],
         rules=["lot_qty == segregation_qty + ok_qty"]),
    dict(form_id="supplier_rework", version=1, name="Supplier Rework Report", family="supplier",
         match=dict(title_any=["SUPPLIER REWORK REPORT"], title_none=["REWORK & REJECTION SHEET", "REWORK AND REJECTION"],
                    columns_any=["DATE", "PART NAME", "SUPPLIER NAME", "LOT QTY", "REWORK QTY", "OK QTY",
                                 "PROBLEM DESCRIPTION", "REMARKS"]),
         header=RECORD_HEADER,
         layout=dict(kind="record", header_rows=3, footer_rows=1, expected_col_lines=9),
         columns=SUPPLIER_COLS_START + [col("lot_qty", "LOT QTY", required=True),
                  col("rework_qty", "REWORK QTY", required=True), col("ok_qty", "OK QTY", required=True),
                  T("problem_description", "PROBLEM DESCRIPTION"), T("remarks", "REMARKS")],
         rules=["rework_qty <= lot_qty", "ok_qty <= lot_qty"]),
    dict(form_id="supplier_rejection", version=1, name="Supplier Rejection Report", family="supplier",
         match=dict(title_any=["SUPPLIER REJECTION REPORT"],
                    title_none=["LOT REJECTION", "LINE REJECTION", "PENETRATION"],
                    columns_any=["DATE", "PART NAME", "SUPPLIER NAME", "LOT QTY", "REJECTION QTY", "OK QTY",
                                 "PROBLEM DESCRIPTION", "REMARKS"]),
         header=RECORD_HEADER,
         layout=dict(kind="record", header_rows=3, footer_rows=1, expected_col_lines=9),
         columns=SUPPLIER_COLS_START + [col("lot_qty", "LOT QTY", required=True),
                  col("rejection_qty", "REJECTION QTY", required=True), col("ok_qty", "OK QTY", required=True),
                  T("problem_description", "PROBLEM DESCRIPTION"), T("remarks", "REMARKS")],
         rules=["lot_qty == rejection_qty + ok_qty"]),
    # press shop, weld shop and the penetration (cutting sample) report share this layout
    dict(form_id="line_rejection", version=1, name="Line Rejection Report",
         match=dict(title_any=["LINE REJECTION REPORT", "REJECTION REPORT PENETRATION", "PENETRATION"],
                    title_none=["SUPPLIER"],
                    columns_any=["S.NO", "ITEM CODE", "MATERIAL PARTICULARS", "UNIT", "REJECTION QTY",
                                 "VENDOR NAME", "REASON / REMARKS"]),
         header={"company": ["COMPANY"], "title": ["TITLE"], "doc_no": ["DOC NO"], "rev_no": ["REV NO"],
                 "rev_date": ["REV DATE"], "org_date": ["ORG DATE"], "shop": ["SHOP", "PENETRATION"],
                 "month": ["MONTH"]},
         layout=dict(kind="record", header_rows=5, footer_rows=1, expected_col_lines=8),
         columns=[col("s_no", "S.NO."), T("item_code", "ITEM CODE", required=True, pattern=r"[A-Z]{2,3}\d{6}"),
                  T("material_particulars", "MATERIAL PARTICULARS"), T("unit", "UNIT"),
                  col("rejection_qty", "REJECTION QTY", required=True), T("vendor_name", "VENDOR NAME"),
                  T("reason_remarks", "REASON / REMARKS")],
         rules=[]),
    dict(form_id="scrap_note", version=1, name="Scrap Note",
         match=dict(title_any=["SCRAP NOTE"],
                    columns_any=["S.NO", "ITEM CODE", "ITEM NAME", "UNIT", "REJECTION QTY", "VENDOR NAME",
                                 "REASON / REMARKS"]),
         header={"company": ["COMPANY"], "title": ["TITLE"], "doc_no": ["DOC NO"], "rev_no": ["REV NO"],
                 "rev_date": ["REV DATE"], "org_date": ["ORG DATE"], "month": ["MONTH"], "period": ["PERIOD"]},
         layout=dict(kind="record", header_rows=4, footer_rows=1, expected_col_lines=8),
         columns=[col("s_no", "S.No."), T("item_code", "ITEM CODE", required=True, pattern=r"WLD\d{6}"),
                  T("item_name", "ITEM NAME"), T("unit", "UNIT"), col("rejection_qty", "REJECTION QTY.", required=True),
                  T("vendor_name", "VENDOR. NAME"), T("reason_remarks", "REASON / REMARKS")],
         rules=[]),
    # ---------------------------------------------------------------- straight pass (block grid)
    dict(form_id="st_pass_monitoring", version=1, name="BS6 Model Component Straight Pass Monitoring Sheet",
         match=dict(title_any=["COMPONENT STRAIGHT PASS MONITORING", "STRAIGHT PASS MONITORING SHEET"],
                    columns_any=["SNO", "PART NAME", "PART NUMBER", "MODEL", "PARAMTER", "A", "B", "TOTAL"]),
         header={"company": ["COMPANY"], "title": ["TITLE"], "month": ["MONTH"]},
         layout=dict(kind="block_grid", header_bands=2, days_per_sheet=10, shifts=["A", "B", "TOTAL"],
                     params=[dict(key="checked_qty", label="CHECKED QTY."), dict(key="ok_qty", label="OK QTY."),
                             dict(key="rej_qty", label="REJ. QTY."), dict(key="rew_qty", label="REW. QTY."),
                             dict(key="total_rework_ok_qty", label="TOTAL REWORK OK QTY"),
                             dict(key="inspection_chipping_pending", label="INSPECTION/CHIPPING PENDING"),
                             dict(key="visual_defects_straight_pass_pct", label="VISUAL DEFECTS STRAIGHT PASS (%)"),
                             dict(key="non_conformity_count", label="NO. OF NON CONFORMITY")]),
         columns=[],
         rules=["checked_qty == ok_qty + rew_qty + rej_qty", "abs(visual_defects_straight_pass_pct - 100 * ok_qty / checked_qty) <= 2"]),
    # ---------------------------------------------------------------- MIS defect matrices
    dict(form_id="mis_fi_visual_pc", version=1, name="FI Visual Inspection 100% (Powder Coating)", family="mis",
         match=dict(title_any=["FI VISUAL INSPECTION"],
                    columns_any=["S.NO.", "DESCRIPTION OF DEFECT", "SARI GUARD-AAH", "SARI GUARD-AAE", "STEP WOMEN-AAH",
                                 "ENGINE GUARD-KVH", "CROSS PIPE AAEL", "TOTAL", "MINOR", "MAJOR", "SCRAP", "B-GRADE/GP"]),
         header=MIS_HEADER, layout=dict(MIS_LAYOUT),
         columns=[L("sr_no", "S.NO."), L("defect", "DESCRIPTION OF DEFECT"),
                  P("sari_guard_aah", "SARI GUARD-AAH"), P("sari_guard_aae", "SARI GUARD-AAE"),
                  P("step_women_aah", "STEP WOMEN-AAH"), P("step_pillion_aae", "STEP PILLION-AAE"),
                  P("sari_guard_aacy", "SARI GUARD-AACY"), P("step_women_aacy", "STEP WOMEN AACY"),
                  P("engine_guard_kvh", "ENGINE GUARD-KVH"), P("cross_pipe_aael", "CROSS PIPE AAEL"),
                  P("handle_abza", "HANDEL ABZA"), P("engine_guard_abza_painted", "ENGINE GUARD-ABZA PAINTED"),
                  P("pedal_brake_abza", "PEDAL BRAKE-ABZA"), P("side_stand_abza", "SIDE STAND ABZA"), total(),
                  disp("for_rework_minor", "MINOR", group="FOR REWORK"), disp("for_rework_major", "MAJOR", group="FOR REWORK"),
                  disp("rejection_scrap", "SCRAP", group="REJECTION"), disp("b_grade_gp", "B-GRADE/GP")],
         rules=[]),
    dict(form_id="mis_plating", version=1, name="Daily MIS Report – Plating", family="mis",
         match=dict(title_any=["DAILY MIS REPORT"], doc_no_any=["/PL"],
                    columns_any=["SR.NO", "DEFECT", "REASON", "CR AAEL", "CR AAD", "EG-KST", "PB-AAE", "PGC", "HANDLE",
                                 "TOTAL", "RENICKEL", "RECHROME", "BUFF", "OK", "B GRADE", "SCRAP"]),
         header=MIS_HEADER, layout=dict(MIS_LAYOUT),
         columns=MIS_LABELS + [P("cr_aael", "CR AAEL"), P("cr_aad", "CR AAD"), P("engine_guard", "EG-KST", "ENGINE GUARD"),
                               P("pedal_brake", "PB-AAE", "PEDAL BRAKE"), P("pgc", "PGC"), P("handle", "HANDLE"), total(),
                               disp("renickel", "RENICKEL"), disp("rechrome", "RECHROME"), disp("buff", "BUFF"),
                               disp("ok", "OK"), disp("b_grade", "B GRADE"), disp("scrap", "SCRAP"),
                               col("grand_total", "TOTAL", role="grand_total")],
         rules=[]),
    dict(form_id="mis_pc_daily_qa", version=1, name="Daily MIS Report – Powder Coating (with disposition)", family="mis",
         match=dict(title_any=["DAILY MIS REPORT"], title_none=["PLATING"],
                    columns_any=["SR.NO", "DEFECT", "REASON", "SG-AAH", "SG-AAE", "SW (AAH)", "SP-AAE", "EG-KVH",
                                 "SIDE STAND", "TOTAL", "REWORK", "MAJOR REWORK", "OK", "B GRADE", "SCRAP"]),
         header=MIS_HEADER, layout=dict(MIS_LAYOUT),
         columns=MIS_LABELS + PC_PARTS + [total(), disp("rework", "REWORK"), disp("major_rework", "MAJOR REWORK"),
                                          disp("ok", "OK"), disp("b_grade", "B GRADE"), disp("scrap", "SCRAP"),
                                          col("grand_total", "TOTAL", role="grand_total")],
         rules=[]),
    dict(form_id="mis_pc_daily", version=1, name="Daily MIS Report – Powder Coating", family="mis",
         match=dict(title_any=["DAILY MIS REPORT"], title_none=["PLATING"],
                    columns_any=["SR.NO", "DEFECT", "REASON", "SG-AAH", "SG-AAE", "SW (AAH)", "SP-AAE", "EG-KVH",
                                 "SIDE STAND", "TOTAL"]),
         header=MIS_HEADER, layout=dict(MIS_LAYOUT),
         columns=MIS_LABELS + PC_PARTS + [total()],
         rules=[]),
    # ---------------------------------------------------------------- IOT weld shop
    dict(form_id="weld_qg2_checksheet", version=1, name="QGFI Check Sheet – Quality Gate-2 (Weld Shop)",
         match=dict(title_any=["QGFI CHECK SHEET", "QUALITY GATE-2", "QUALITY GATE 2", "QGFI-WELD SHOP"],
                    title_none=["POWDER COATING"], doc_no_any=["QG-02", "QG-2", "/39/"],
                    columns_any=["S.NO", "OPERATION NAME", "DEFECT'S DESCRIPTION", "DESCRIPTION", "WELDER NAME",
                                 "7 TO 9", "9 TO 11", "11 TO 1", "1 TO 3", "3 TO 5", "5 TO 7", "TOTAL", "OBSERVATION"]),
         header={"company": ["COMPANY"], "title": ["TITLE"], "doc_no": ["DOC NO"], "part_name": ["PART NAME"],
                 "shift": ["SHIFT"], "line": ["LINE", "LINE NO"], "date": ["DATE"]},
         # rows from the S.No column (also holds the summary labels) + the DESCRIPTION column
         layout=dict(kind="matrix", row_id="sr_no", row_name_key="defect", row_strip_cols=[[0, 1], [2, 3]], min_rows=15,
                     header_terms=["S NO", "SNO", "DEFECT", "DESCRIPTION", "OPERATION"],
                     footer_terms=["NAME INSPECTOR", "NAME & SIGNATURE", "PREPARED BY", "VERIFIED BY", "APPROVED BY"],
                     row_sum_on_summary=True),
         columns=[L("sr_no", "S.No", "S.NO"), L("operation", "OPERATION NAME"),
                  L("defect", "DEFECT'S DESCRIPTION", "DESCRIPTION"),
                  T("welder_name", "OPERATOR /WELDER NAME", "WELDER NAME", role="text"),
                  P("slot_07_09", "7 TO 9", group="OBSERVATION", fmt="tally_with_count"),
                  P("slot_09_11", "9 TO 11", group="OBSERVATION", fmt="tally_with_count"),
                  P("slot_11_01", "11 TO 1", group="OBSERVATION", fmt="tally_with_count"),
                  P("slot_01_03", "1 TO 3", group="OBSERVATION", fmt="tally_with_count"),
                  P("slot_03_05", "3 TO 5", group="OBSERVATION", fmt="tally_with_count"),
                  P("slot_05_07", "5 TO 7", group="OBSERVATION", fmt="tally_with_count"),
                  total()],
         summary_rows=[dict(key="total_production", labels=["TOTAL PRODUCTION"]),
                       dict(key="ok_qty", labels=["OK QTY (FIRST TIME RIGHT)", "OK QTY"]),
                       dict(key="defect_qty", labels=["DEFECT QTY (REJECTION)", "DEFECT QTY"]),
                       dict(key="rework_ok", labels=["REWORK OK"]),
                       dict(key="straight_pass", labels=["STRAIGHT PASS"]),
                       dict(key="scrap_qty", labels=["SCRAP QTY"]),
                       dict(key="pending_qty", labels=["INSPECTION/CHIPPING PENDING QTY"])],
         summary_rules=["total_production == ok_qty + defect_qty"],
         rules=[]),
    dict(form_id="weld_rework_rejection", version=1, name="Weld Shop Rework & Rejection Sheet",
         match=dict(title_any=["WELDSHOP REWORK & REJECTION SHEET", "REWORK & REJECTION SHEET", "REWORK AND REJECTION SHEET"],
                    columns_any=["LINE / PART NAME", "REWORK", "GRINDER", "CUTTING", "REJECTION",
                                 "CHIPPING/ FINAL PENDING", "QUALITY PENDING", "QUALITY HOLD", "INSPECTOR SIGH"]),
         header={"company": ["COMPANY"], "title": ["TITLE"], "date": ["DATE"], "shift": ["SHIFT"]},
         layout=dict(kind="matrix", row_id="label", row_name_key="part_name", row_strip_cols=[0, 1], min_rows=15,
                     header_terms=["LINE", "PART NAME"], require_total=False, allow_gaps=True,
                     blocks=dict(split_labels=["LINE / PART NAME"], names=["A", "B"]),
                     fixed_rows=REWORK_PARTS, footer_terms=["QA SIGN", "PROD SIGN", "PROD. SIGN"]),
         columns=[L("part_name", "LINE / PART NAME"), col("rework", "REWORK", type="qty_text", role="value"),
                  col("grinder", "GRINDER", type="qty_text", role="value"),
                  col("cutting", "CUTTING", type="qty_text", role="value"),
                  col("rejection", "REJECTION", type="qty_text", role="value"),
                  col("chipping_final_pending", "CHIPPING/ FINAL PENDING", type="qty_text", role="value"),
                  col("quality_pending", "QUALITY PENDING", type="qty_text", role="value"),
                  col("quality_hold", "QUALITY HOLD", type="qty_text", role="value"),
                  T("inspector_sign", "INSPECTOR SIGH", "INSPECTOR SIGN", role="ignore")],
         rules=[]),
]

KINDS = ("record", "matrix", "block_grid")
COLUMN_TYPES = ("int", "number", "text", "date", "qty_text")
ROLES = ("label", "text", "part", "total", "disposition", "grand_total", "value", "ignore")

def check_spec(spec):
    problems = []
    for k in ("form_id", "name", "layout"):
        if not spec.get(k):
            problems.append(f"missing {k!r}")
    kind = (spec.get("layout") or {}).get("kind")
    if kind not in KINDS:
        problems.append(f"layout.kind must be one of {KINDS}, not {kind!r}")
    keys = [c.get("key") for c in spec.get("columns") or []]
    if len(keys) != len(set(keys)):
        problems.append("duplicate column keys")
    for c in spec.get("columns") or []:
        if not c.get("key") or not c.get("labels"):
            problems.append(f"column without key / labels: {c}")
        if c.get("type", "int") not in COLUMN_TYPES:
            problems.append(f"column {c.get('key')}: type {c.get('type')!r}")
        if c.get("role", "value") not in ROLES:
            problems.append(f"column {c.get('key')}: role {c.get('role')!r}")
    if kind in ("record", "matrix") and not spec.get("columns"):
        problems.append("no columns")
    if kind == "block_grid" and not (spec.get("layout") or {}).get("params"):
        problems.append("block_grid needs layout.params")
    for r in (spec.get("rules") or []) + (spec.get("summary_rules") or []):
        try:
            ast.parse(r, mode="eval")
        except SyntaxError as e:
            problems.append(f"rule {r!r}: {e}")
    return problems

def load_forms(write_builtin=True):
    FORMS_DIR.mkdir(parents=True, exist_ok=True)
    if write_builtin:
        for spec in BUILTIN_FORMS:
            p = FORMS_DIR / f"{spec['form_id']}.json"
            if not p.exists():
                p.write_text(json.dumps(spec, indent=2, ensure_ascii=False), encoding="utf-8")
    forms = {}
    for p in sorted(FORMS_DIR.glob("*.json")):
        try:
            spec = json.loads(p.read_text(encoding="utf-8"))
        except Exception as e:
            print(f"⚠ {p.name}: not valid JSON ({e}) - skipped")
            continue
        problems = check_spec(spec)
        if problems:
            print(f"⚠ {p.name}: {'; '.join(problems)} - skipped")
            continue
        forms[spec["form_id"]] = spec
    return forms

FORMS = load_forms()
print(f"{len(FORMS)} form specs:", ", ".join(FORMS))

## 4. Image preprocessing and table geometry

In [ ]:
class Debug:
    """Numbered debug images for one page."""
    def __init__(self, root):
        self.root = Path(root); self.root.mkdir(parents=True, exist_ok=True); self.n = 0
    def save(self, name, img):
        path = self.root / f"{self.n:02d}_{name}.png"; self.n += 1
        cv2.imwrite(str(path), img)
        return path

def show(img, title="", width=14):
    if img is None:
        return
    h, w = img.shape[:2]
    plt.figure(figsize=(width, max(1.0, width * h / w)))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img, cmap="gray")
    plt.title(title, fontsize=10); plt.axis("off"); plt.show(); plt.close()

def load_page(page):
    # rendering (not extracting the JPEG) applies the PDF placement matrix: un-flips upside-down / sideways scans
    infos = page.get_image_info()
    if infos:
        info = max(infos, key=lambda i: i["width"] * i["height"])
        zoom = info["width"] / max(info["bbox"][2] - info["bbox"][0], 1)
    else:
        zoom = 300 / 72
    zoom = min(max(zoom, 150 / 72), 400 / 72)
    pix = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), colorspace=pymupdf.csRGB, alpha=False)
    rgb = np.frombuffer(pix.samples, np.uint8).reshape(pix.h, pix.stride)[:, :pix.w * 3].reshape(pix.h, pix.w, 3)
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), dict(zoom=round(zoom, 3), scanned=bool(infos))

def normalise_width(img):
    width = PORTRAIT_WIDTH if img.shape[0] >= img.shape[1] else LANDSCAPE_WIDTH
    s = width / img.shape[1]
    return img if abs(s - 1) < 0.01 else cv2.resize(img, None, fx=s, fy=s,
                                                     interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)

def to_gray(img): return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def binarize(gray):  # ink = white
    return cv2.adaptiveThreshold(~gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 15, -2)

def horizontal_lines(bw):
    return cv2.morphologyEx(bw, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (bw.shape[1] // 25, 1)))

def vertical_lines(bw):
    return cv2.morphologyEx(bw, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, bw.shape[0] // 40)))

def deskew(img):
    small = cv2.resize(img, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)    # 4x faster, same angle
    h = horizontal_lines(binarize(to_gray(small)))
    segs = cv2.HoughLinesP(h, 1, np.pi / 1800, 100, minLineLength=small.shape[1] // 4, maxLineGap=10)
    if segs is None:
        return img, 0.0
    angles = [np.degrees(np.arctan2(y2 - y1, x2 - x1)) for x1, y1, x2, y2 in np.asarray(segs).reshape(-1, 4)]
    angles = [a for a in angles if abs(a) < 5]
    angle = float(np.median(angles)) if angles else 0.0
    M = cv2.getRotationMatrix2D((img.shape[1] / 2, img.shape[0] / 2), angle, 1)
    return cv2.warpAffine(img, M, img.shape[1::-1], flags=cv2.INTER_CUBIC, borderValue=(255, 255, 255)), angle

def flatten_lighting(img):
    bg = cv2.morphologyEx(img, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (51, 51)))
    return cv2.divide(img, bg, scale=255)

def line_positions(mask, axis, frac):
    prof = mask.sum(axis=axis) / 255
    if prof.max() == 0:
        return []
    idx = np.where(prof > frac * prof.max())[0]
    groups = np.split(idx, np.where(np.diff(idx) > 5)[0] + 1)
    return [int(g.mean()) for g in groups if len(g)]

def filter_columns(gray, rows, cands):
    # a printed line is dark with equally bright paper on both sides; a shadow edge is bright on one side only
    kept, report = [], []
    strip = gray[rows[0]:rows[-1]] if len(rows) >= 2 else gray
    for k, x in enumerate(cands):
        line  = strip[:, max(x - 2, 0):x + 3].min(axis=1).mean()
        left  = strip[:, max(x - 15, 0):max(x - 8, 1)].mean()
        right = strip[:, x + 8:x + 15].mean() if x + 15 <= strip.shape[1] else 255.0
        border = k in (0, len(cands) - 1)
        ok = (min(left, right) - line > 25) and (border or abs(left - right) < 12)
        report.append(dict(x=x, line=round(float(line)), left=round(float(left)), right=round(float(right)), kept=bool(ok)))
        if ok:
            kept.append(x)
    return kept, report

def ink_mask(gray, h, v):
    ink = ((gray < 160) * 255).astype(np.uint8)
    ink = cv2.subtract(ink, cv2.dilate(h | v, np.ones((7, 7), np.uint8)))
    n, lab, st, _ = cv2.connectedComponentsWithStats(ink)
    return (np.isin(lab, np.where(st[:, 4] >= 25)[0][1:]) * 255).astype(np.uint8)

def with_margin(img, m=IMAGE_MARGIN):
    return cv2.copyMakeBorder(img, m, m, m, m, cv2.BORDER_CONSTANT, value=(255, 255, 255))

def find_line_pieces(mask, min_frac=0.4, max_gap=80, min_fill=0.8):
    """Horizontal ruled lines as connected pieces (tolerant to slight tilt / page curl). A row of handwritten dashes
    is dropped because it is not continuous (fill < min_fill). Returns [{y, x0, x1}] sorted by y."""
    m = cv2.dilate(mask, cv2.getStructuringElement(cv2.MORPH_RECT, (31, 5)))
    n, lab, st, _ = cv2.connectedComponentsWithStats(m)
    pieces = []
    for i in range(1, n):
        x, y, w, h = st[i, :4]
        if w < 60:
            continue
        rows = (lab[y:y + h, x:x + w] == i).sum(axis=1)
        pieces.append(dict(y=int(y + np.argmax(rows)), x0=int(x), x1=int(x + w), lo=int(y), hi=int(y + h)))
    merged = []
    for p in sorted(pieces, key=lambda p: p["x0"]):
        for q in merged:
            if abs(q["y"] - p["y"]) <= 12 and p["x0"] - q["x1"] <= max_gap:
                q.update(x0=min(q["x0"], p["x0"]), x1=max(q["x1"], p["x1"]), lo=min(q["lo"], p["lo"]), hi=max(q["hi"], p["hi"]))
                break
        else:
            merged.append(dict(p))
    for q in merged:
        q["fill"] = float((mask[q["lo"]:q["hi"] + 1, q["x0"]:q["x1"]].max(axis=0) > 0).mean())
    merged = [q for q in merged if q["fill"] >= min_fill]
    if not merged:
        return []
    longest = max(q["x1"] - q["x0"] for q in merged)
    return sorted([q for q in merged if q["x1"] - q["x0"] >= min_frac * longest], key=lambda q: q["y"])

def uniform_run(pitches, tol=0.2, min_len=4):
    """Longest run of consecutive row heights within ±tol of the run's median -> (first, last) band index."""
    best = None
    for s in range(len(pitches)):
        e = s
        while e + 1 < len(pitches):
            ref = float(np.median(pitches[s:e + 2]))
            if all(abs(p - ref) <= tol * ref for p in pitches[s:e + 2]):
                e += 1
            else:
                break
        if e - s + 1 >= min_len and (best is None or e - s > best[1] - best[0]):
            best = (s, e)
    return best

def prepare_page(page, dbg):
    img, info = load_page(page);                 dbg.save("rendered", img)
    img = normalise_width(img)
    img, angle = deskew(img);                    dbg.save("deskewed", img)
    raw_gray = to_gray(img)                                   # before flattening: judges line contrast
    img = flatten_lighting(img);                 dbg.save("flattened", img)
    return img, raw_gray, angle, info

def basic_grid(img):
    gray = to_gray(img)
    bw = binarize(gray)
    h, v = horizontal_lines(bw), vertical_lines(bw)
    return gray, bw, h, v, line_positions(h, 1, 0.5), line_positions(v, 0, 0.3)

## 5. Model (Qwen3-VL) with token probabilities
`TokenRecorder` is a logits processor: at every generation step it stores the top-`TOP_K` token probabilities
(a few numbers per step, never the full vocabulary). With greedy decoding the chosen token is the top one, so its
probability is known for every generated token. `decode_with_offsets` maps every token to its characters in the
answer, and `value_confidences` maps every JSON value to the tokens that spell it.

In [ ]:
# Weights are downloaded once per Colab runtime. Re-running this cell keeps a model already on the GPU.
if globals().get("LOADED_MODEL_ID") != MODEL_ID:
    model = processor = LOADED_MODEL_ID = None
if not DRY_RUN and model is not None:
    print("model already loaded:", MODEL_ID, f"| GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
elif not DRY_RUN:
    from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
    free_cuda = getattr(torch.cuda, "empty_cache", None)
    import gc; gc.collect(); free_cuda and free_cuda()
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model = Qwen3VLForConditionalGeneration.from_pretrained(MODEL_ID, dtype=dtype, device_map="auto",
                                                            attn_implementation="sdpa")
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model.eval()
    LOADED_MODEL_ID = MODEL_ID
    print("loaded", MODEL_ID, dtype, f"| weights {torch.cuda.memory_allocated() / 1e9:.1f} GB")
else:
    print("DRY_RUN: model not loaded")

gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if HAS_CUDA else 0
if MAX_IMAGE_PIXELS is None:
    MAX_IMAGE_PIXELS = 1_000_000 if gpu_gb < 20 else 2_000_000
if processor is not None:          # the processor must not upscale our crops beyond the cap either
    processor.image_processor.size = {"shortest_edge": 64 * 32 * 32, "longest_edge": MAX_IMAGE_PIXELS}
    for attr, val in (("max_pixels", MAX_IMAGE_PIXELS), ("min_pixels", 64 * 32 * 32)):
        if hasattr(processor.image_processor, attr):
            setattr(processor.image_processor, attr, val)
print(f"GPU {gpu_gb:.0f} GB | image cap {MAX_IMAGE_PIXELS / 1e6:.1f} MP")

In [ ]:
def bytes_to_unicode():
    """The byte <-> printable-character table of byte-level BPE tokenizers (GPT-2 / Qwen)."""
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs, n = bs[:], 0
    for b in range(256):
        if b not in bs:
            bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, map(chr, cs)))

BYTE_DECODER = {c: b for b, c in bytes_to_unicode().items()}

def decode_with_offsets(ids, tokenizer):
    """-> (text, [(start, end) or None per token]). Special tokens (end of turn, padding) get None.
    Same text as tokenizer.decode(ids, skip_special_tokens=True); byte offsets are turned into character offsets,
    so a character split over two tokens (e.g. ①) belongs to both."""
    special = set(getattr(tokenizer, "all_special_ids", []) or [])
    pieces = tokenizer.convert_ids_to_tokens(list(ids))
    buf, bspans = bytearray(), []
    for tid, pc in zip(ids, pieces):
        if tid in special or pc is None:
            bspans.append(None); continue
        try:
            b = bytes(BYTE_DECODER[c] for c in pc)
        except KeyError:                                        # added token that is not byte-level
            b = tokenizer.decode([tid]).encode("utf-8")
        bspans.append((len(buf), len(buf) + len(b))); buf += b
    chars, b2c, i = [], [0] * (len(buf) + 1), 0
    while i < len(buf):
        lead = buf[i]
        n = 1 if lead < 0x80 else 2 if lead >> 5 == 0b110 else 3 if lead >> 4 == 0b1110 else 4 if lead >> 3 == 0b11110 else 1
        try:
            ch = bytes(buf[i:i + n]).decode("utf-8")
        except UnicodeDecodeError:
            ch, n = "�", 1
        for j in range(i, min(i + n, len(buf))):
            b2c[j] = len(chars)
        chars.append(ch); i += n
    b2c[len(buf)] = len(chars)
    spans = [None if s is None else (b2c[s[0]], b2c[s[1] - 1] + 1 if s[1] > s[0] else b2c[s[0]]) for s in bspans]
    return "".join(chars), spans

if torch is not None:
    from transformers import LogitsProcessor, LogitsProcessorList

    class TokenRecorder(LogitsProcessor):
        """Stores the top-k probabilities of every generation step; changes nothing."""
        def __init__(self, k=TOP_K):
            self.k, self.ids, self.probs = k, [], []
        def __call__(self, input_ids, scores):
            p = torch.softmax(scores.float(), dim=-1).topk(self.k, dim=-1)
            self.ids.append(p.indices); self.probs.append(p.values)
            return scores
        def stacked(self):           # -> (batch, steps, k) tensors on the CPU
            if not self.ids:
                return None, None
            return torch.stack(self.ids, 1).cpu(), torch.stack(self.probs, 1).cpu()

def token_table(new_ids, top_ids, top_p, tokenizer):
    """One generated sequence -> (text, tokens) with tokens = [{s, e, p, alts: [(piece, p)]}]."""
    ids = [int(x) for x in new_ids]
    text, spans = decode_with_offsets(ids, tokenizer)
    toks = []
    for t, (tid, sp) in enumerate(zip(ids, spans)):
        if sp is None or top_ids is None or t >= top_ids.shape[0]:
            continue
        row_ids, row_p = top_ids[t].tolist(), top_p[t].tolist()
        p = row_p[row_ids.index(tid)] if tid in row_ids else None
        alts = [(tokenizer.decode([i]), round(q, 4)) for i, q in zip(row_ids, row_p) if i != tid and q >= ALT_MIN_P]
        toks.append(dict(s=sp[0], e=sp[1], p=None if p is None else round(p, 4), alts=alts))
    return text, toks

# ---- spans of every value in a JSON answer ---------------------------------------------------------
class _Spans:
    """Tolerant JSON reader that records the character span of every scalar value: {path: (start, end, kind)}.
    Strings: the span inside the quotes. Stops quietly at the first error (spans found so far are kept)."""
    def __init__(self, text):
        self.t, self.i, self.out = text, 0, {}
    def ws(self):
        while self.i < len(self.t) and self.t[self.i] in " \t\r\n":
            self.i += 1
    def value(self, path):
        self.ws()
        c = self.t[self.i]
        if c == "{":
            self.i += 1
            while True:
                self.ws()
                if self.t[self.i] == "}":
                    self.i += 1; return
                key = self.string(None)
                self.ws(); assert self.t[self.i] == ":"; self.i += 1
                self.value(path + (key,))
                self.ws()
                if self.t[self.i] == ",":
                    self.i += 1
        elif c == "[":
            self.i += 1; k = 0
            while True:
                self.ws()
                if self.t[self.i] == "]":
                    self.i += 1; return
                self.value(path + (k,)); k += 1
                self.ws()
                if self.t[self.i] == ",":
                    self.i += 1
        elif c == '"':
            self.string(path)
        else:
            m = re.compile(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?|true|false|null").match(self.t, self.i)
            assert m
            self.out[path] = (m.start(), m.end(), "null" if m.group() == "null" else "scalar")
            self.i = m.end()
    def string(self, path):
        assert self.t[self.i] == '"'
        s = self.i + 1; j = s
        while self.t[j] != '"':
            j += 2 if self.t[j] == "\\" else 1
        self.i = j + 1
        if path is not None:
            self.out[path] = (s, j, "string")
        return json.loads(self.t[s - 1:j + 1])

def json_spans(text):
    starts = [i for i in (text.find("{"), text.find("[")) if i >= 0]
    if not starts:
        return {}
    sp = _Spans(text); sp.i = min(starts)
    try:
        sp.value(())
    except (AssertionError, IndexError, ValueError):
        pass
    return sp.out

def value_confidences(text, toks):
    """{path: {"model": p, "alternatives": [{"value": text, "p": q}]}} for every value of a JSON answer.
    model = lowest probability among the value's tokens; alternatives = the weakest token swapped for the model's
    other candidates (only when that token lies inside the value). A null value gets the probability of "null"."""
    out = {}
    for path, (a, b, kind) in json_spans(text).items():
        inside = [t for t in toks if t["s"] < b and t["e"] > a and t["p"] is not None
                  and text[max(t["s"], a):min(t["e"], b)].strip(' "')]
        if not inside:
            continue
        weak = min(inside, key=lambda t: t["p"])
        alts = []
        if a <= weak["s"] and weak["e"] <= b:
            for piece, q in weak["alts"]:
                cand = (text[a:weak["s"]] + piece + text[weak["e"]:b]).strip()
                if cand and cand != text[a:b].strip():
                    alts.append(dict(value=cand, p=q))
        out[path] = dict(model=weak["p"], alternatives=alts[:3], null=kind == "null")
    return out

In [ ]:
def parse_json(text):
    if not text or not text.strip():
        return None, "empty output"
    t = re.sub(r"```(?:json)?", "", text).strip()
    starts = [i for i in (t.find("{"), t.find("[")) if i >= 0]
    if not starts:
        return None, "no JSON in output"
    s = t[min(starts):max(t.rfind("}"), t.rfind("]")) + 1]
    err = None
    for cand in (s, re.sub(r",\s*([}\]])", r"\1", s)):
        try:
            return json.loads(cand), None
        except json.JSONDecodeError as e:
            err = e
    return None, f"invalid JSON ({err}) - output cut off? raise MAX_NEW_TOKENS"

def _generate(image_paths, prompt, max_new_tokens):
    msgs = [[{"role": "user", "content": [{"type": "image", "image": str(p)}, {"type": "text", "text": prompt}]}]
            for p in image_paths]
    batched = len(image_paths) > 1
    if batched:
        processor.tokenizer.padding_side = "left"
    inputs = processor.apply_chat_template(msgs if batched else msgs[0], tokenize=True, add_generation_prompt=True,
                                           return_dict=True, return_tensors="pt", padding=batched).to(model.device)
    rec = TokenRecorder()
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             logits_processor=LogitsProcessorList([rec]))
    new = out[:, inputs["input_ids"].shape[1]:]
    top_ids, top_p = rec.stacked()
    pad = processor.tokenizer.pad_token_id
    patch = getattr(processor.image_processor, "patch_size", 16)
    secs = round((time.time() - t0) / len(image_paths), 2)
    res = []
    for i in range(len(image_paths)):
        text, toks = token_table(new[i], None if top_ids is None else top_ids[i],
                                 None if top_p is None else top_p[i], processor.tokenizer)
        grid = [int(x) for x in inputs["image_grid_thw"][i].tolist()]
        n_out = int((new[i] != pad).sum()) if batched else int(new.shape[1])
        res.append(dict(raw=text, tokens=toks, seconds=secs,
                        input_tokens=int(inputs["attention_mask"][i].sum()), output_tokens=n_out, grid_thw=grid,
                        model_view_wh=(grid[2] * patch, grid[1] * patch), hit_token_limit=n_out >= max_new_tokens,
                        batch=len(image_paths)))
    return res

class PageCtx:
    """Folders + call log for one page."""
    def __init__(self, pdf_rel, page_no, root):
        self.pdf_rel, self.page_no, self.dir = pdf_rel, page_no, Path(root)
        self.inputs = self.dir / "model_inputs"; self.inputs.mkdir(parents=True, exist_ok=True)
        self.view = self.dir / "model_view"
        self.dbg = Debug(self.dir / "debug")
        self.calls, self.shown = [], 0
        self.overlay, self.extra = None, {}
        (self.dir / "model_io.jsonl").unlink(missing_ok=True)

def cap_pixels(img, max_pixels):
    px = img.shape[0] * img.shape[1]
    if px <= max_pixels:
        return img
    s = (max_pixels / px) ** 0.5
    return cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)

def free_gpu():
    import gc
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()

EMPTY_RESULT = dict(raw="", tokens=[], seconds=0.0, input_tokens=None, output_tokens=None, grid_thw=None,
                    model_view_wh=None, hit_token_limit=False)

def call_model(ctx, name, img, prompt, max_new_tokens=MAX_NEW_TOKENS):
    """One image -> (parsed JSON, call record). The record holds `conf`: {json path: value confidence}.
    The saved input is exactly what the model gets. On CUDA out-of-memory: retry at 70 % / 50 % of the pixels."""
    path = ctx.inputs / f"{len(ctx.calls):02d}_{name}.png"
    sent = cap_pixels(img, MAX_IMAGE_PIXELS)
    cv2.imwrite(str(path), sent)
    if DRY_RUN:
        return log_call(ctx, name, path, sent, prompt, dict(EMPTY_RESULT), None, "dry run")
    r, parsed, err = dict(EMPTY_RESULT), None, None
    for scale in (1.0, 0.7, 0.5):
        if scale < 1.0:
            sent = cap_pixels(img, int(MAX_IMAGE_PIXELS * scale))
            cv2.imwrite(str(path), sent)
        oom = False
        try:
            r = _generate([path], prompt, max_new_tokens)[0]
            parsed, err = parse_json(r["raw"])
            if scale < 1.0:
                err = (err + "; " if err else "") + f"ran at {int(scale * 100)} % pixels after out-of-memory"
        except Exception as e:                               # one failed call must not stop the run
            oom = "out of memory" in str(e).lower()
            err = (f"CUDA out of memory at {sent.shape[1]}x{sent.shape[0]}" if oom
                   else f"model error: {type(e).__name__}: {str(e)[:300]}")
        if not oom:
            break
        free_gpu()
    return log_call(ctx, name, path, sent, prompt, r, parsed, err)

def log_call(ctx, name, path, img, prompt, r, parsed, err):
    view = None
    if SAVE_MODEL_VIEW and r.get("model_view_wh"):
        ctx.view.mkdir(parents=True, exist_ok=True)
        view = ctx.view / path.name
        cv2.imwrite(str(view), cv2.resize(img, tuple(r["model_view_wh"]), interpolation=cv2.INTER_AREA))
    conf = value_confidences(r["raw"], r.get("tokens") or []) if r.get("raw") else {}
    rec = dict(name=name, image=str(path), image_wh=(img.shape[1], img.shape[0]), prompt=prompt,
               **{k: v for k, v in r.items() if k != "tokens"}, parsed=parsed, error=err,
               model_view=str(view) if view else None)
    with open(ctx.dir / "model_io.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(dict(rec, conf={".".join(map(str, k)): v for k, v in conf.items()}),
                           ensure_ascii=False, default=str) + "\n")
    rec["conf"] = conf
    ctx.calls.append(rec)
    if VERBOSE:
        ans = re.sub(r"\s+", " ", r["raw"])[:100]
        weakest = min((v["model"] for v in conf.values() if not v.get("null")), default=None)
        print(f"      · {name:16s} img {img.shape[1]}x{img.shape[0]} tok {r.get('input_tokens')}/{r.get('output_tokens')} "
              f"{r['seconds']:5.1f}s min-conf {weakest if weakest is not None else '-'} "
              f"{'⚠ ' + err if err and not DRY_RUN else ''} {ans}")
    if SHOW_INLINE and ctx.shown < SHOW_MAX_IMAGES and name not in ("header", "skeleton", "mis_locate", "mis_locate_header"):
        ctx.shown += 1
        show(img, f"{ctx.pdf_rel} p{ctx.page_no} · model input {path.name}", width=12)
    return parsed, rec

BATCH_OK = True

def call_model_batch(ctx, names, imgs, prompt, max_new_tokens=MAX_NEW_TOKENS):
    """Same as call_model for several images that share one prompt; falls back to one-by-one on any problem."""
    global BATCH_OK
    if DRY_RUN or ROW_BATCH <= 1 or len(imgs) <= 1 or not BATCH_OK:
        return [call_model(ctx, n, im, prompt, max_new_tokens) for n, im in zip(names, imgs)]
    paths, sents = [], []
    for n, im in zip(names, imgs):
        p = ctx.inputs / f"{len(ctx.calls) + len(paths):02d}_{n}.png"
        sent = cap_pixels(im, MAX_IMAGE_PIXELS)
        cv2.imwrite(str(p), sent)
        paths.append(p); sents.append(sent)
    rs, problem = None, None
    try:
        rs = _generate(paths, prompt, max_new_tokens)
    except Exception as e:
        problem = f"{type(e).__name__}: {str(e)[:200]}"
    if rs is None:
        free_gpu()
        if "out of memory" in (problem or "").lower():
            h = len(imgs) // 2                                  # smaller batches first: 4 -> 2 -> 1
            if VERBOSE:
                print(f"      ⚠ out of memory with {len(imgs)} images - trying batches of {h}")
            return (call_model_batch(ctx, names[:h], imgs[:h], prompt, max_new_tokens)
                    + call_model_batch(ctx, names[h:], imgs[h:], prompt, max_new_tokens))
        BATCH_OK = False
        print(f"      ⚠ batched generation not available ({problem}) - continuing one image at a time")
        return [call_model(ctx, n, im, prompt, max_new_tokens) for n, im in zip(names, imgs)]
    return [log_call(ctx, n, p, sent, prompt, r, *parse_json(r["raw"]))
            for n, p, sent, r in zip(names, paths, sents, rs)]

## 6. Text, numbers and cells
Every value in the output is a **cell**:
`{"value", "raw", "source", "confidence": {"model", "final", "agreement", "alternatives"}, "checks", "status"}`.
`value` is normalised (int, ISO date, trimmed text), `raw` is exactly what the model wrote.

In [ ]:
DASH_RE = re.compile(r"^[\-—–~_=\s]+$")
DATE_RE = re.compile(r"^\s*(\d{1,2})\s*[/.\-]\s*(\d{1,2})\s*[/.\-]\s*(\d{2,4})\s*$")
NO_COUNT_WORDS = {"NC", "NO COUNT", "NO_COUNT", "STROKES", "TALLY"}

def norm_text(s):
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9%/]+", " ", str(s or "").upper())).strip()

def slug(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s or "").lower()).strip("_") or "col"

def fuzzy_contains(text, phrase, thr=0.85):
    """True if every word of `phrase` appears, in order, in `text` - each word may be slightly misread
    ("REPOR" for "REPORT"). Words written together ("WELDSHOP") or split are matched when the letters agree almost
    completely. A phrase never matches on part of its words: "REJECTION REPORT" does NOT match "LINE REJECTION"."""
    tw, pw = text.split(), phrase.split()
    if not pw:
        return False
    if f" {phrase} " in f" {text} ":
        return True
    n = len(pw)
    word_thr = min(thr, 0.8) if n > 1 else thr
    ratio = lambda a, b: difflib.SequenceMatcher(None, a, b).ratio()
    def word_ok(t, p, j):
        if ratio(t, p) >= word_thr:
            return True
        return len(p) >= 3 and ((j == 0 and n > 1 and t.endswith(p)) or (j == n - 1 and n > 1 and t.startswith(p)))
    for i in range(len(tw) - n + 1):
        if all(word_ok(tw[i + j], pw[j], j) for j in range(n)):
            return True
    joined = "".join(pw)
    for k in {max(n - 1, 1), n, n + 1}:
        for i in range(len(tw) - k + 1):
            if ratio("".join(tw[i:i + k]), joined) >= max(thr, 0.92):
                return True
    return False

def phrase_score(text, phrase):
    """1.0 when the phrase is found (fuzzy), else the best similarity of any window of the same length, squared
    (a half-similar title should count for little)."""
    t, p = norm_text(text), norm_text(phrase)
    if not t or not p:
        return 0.0
    if fuzzy_contains(t, p):
        return 1.0
    tw, n = t.split(), len(p.split())
    best = 0.0
    for k in {max(n - 1, 1), n, n + 1}:
        for i in range(max(len(tw) - k + 1, 1)):
            best = max(best, difflib.SequenceMatcher(None, " ".join(tw[i:i + k]), p).ratio())
    return round(best ** 2, 3)

def any_term(label, terms, thr=0.8):
    return any(fuzzy_contains(label, norm_text(t), thr) for t in terms)

def to_int(v):
    if v is None or isinstance(v, bool): return None
    if isinstance(v, int): return v
    if isinstance(v, float): return int(v) if v.is_integer() else None
    s = re.sub(r"[,\s]", "", str(v))
    return int(s) if re.fullmatch(r"-?\d+", s) else None

def to_num(v):
    """number cell: dash -> 0, empty -> None, '82%' -> 82, unreadable -> 'INVALID'."""
    if v is None or isinstance(v, bool): return None
    if isinstance(v, (int, float)): return int(v) if float(v).is_integer() else float(v)
    s = str(v).strip().replace(",", "").replace("%", "").strip()
    if s == "" or s.lower() in ("null", "none"): return None
    if DASH_RE.match(s) or s.lower() == "nil": return 0
    try:
        f = float(s); return int(f) if f.is_integer() else f
    except ValueError:
        return "INVALID"

def iso_date(s):
    m = DATE_RE.match(str(s or ""))
    if not m:
        return None
    d, mo, y = (int(x) for x in m.groups())
    y = y + 2000 if y < 100 else y
    try:
        return time.strftime("%Y-%m-%d", time.strptime(f"{y}-{mo}-{d}", "%Y-%m-%d"))
    except ValueError:
        return None

def is_numeric_type(c):
    return c.get("type", "int") in ("int", "number")

def make_cell(raw, col_spec, conf=None):
    """raw model value + column spec + token confidence -> cell (status is set later by finalize_cell)."""
    typ, fmt = col_spec.get("type", "int"), col_spec.get("cell_format", "plain")
    rawtxt = None if raw is None else str(raw).strip()
    cell = dict(value=None, raw=rawtxt, source="empty", status=None, checks={},
                confidence=dict(model=None if conf is None or conf.get("null") else conf.get("model"),
                                final=None, agreement=None,
                                alternatives=[] if conf is None else list(conf.get("alternatives") or [])))
    if rawtxt in (None, ""):
        cell["status"] = "empty"
        return cell
    if fmt == "tally_with_count" and (norm_text(rawtxt) in NO_COUNT_WORDS or re.fullmatch(r"[|Il1/\\\s]+", rawtxt) and len(rawtxt) > 1 and not rawtxt.isdigit()):
        cell.update(source="no_count_written", status="no_count")
        return cell
    if typ in ("int", "number"):
        v = to_num(rawtxt)
        if v == 0 and DASH_RE.match(rawtxt):
            cell.update(value=0, source="dash", status="empty")
        elif v == "INVALID":
            cell.update(value=None, source="unreadable", status="low")
        else:
            cell.update(value=v, source="count" if fmt == "tally_with_count" else "digit")
            if typ == "int" and isinstance(v, float):
                cell["status"] = "low"
    elif typ == "date":
        cell.update(value=iso_date(rawtxt) or rawtxt, source="text")
        if iso_date(rawtxt) is None:
            cell["checks"]["date_format"] = "fail"
    elif typ == "qty_text":
        if DASH_RE.match(rawtxt):
            cell.update(value=0, source="dash", status="empty")
        else:
            v = to_num(rawtxt)
            cell.update(value=v if isinstance(v, (int, float)) else rawtxt, source="digit" if isinstance(v, (int, float)) else "text")
    else:
        if DASH_RE.match(rawtxt):
            cell.update(value=None, source="dash", status="empty")
        else:
            cell.update(value=rawtxt, source="text")
    if col_spec.get("pattern") and isinstance(cell["value"], str):
        ok = re.fullmatch(col_spec["pattern"], cell["value"].replace(" ", "").upper()) is not None
        cell["checks"]["pattern"] = "pass" if ok else "fail"
    for a in cell["confidence"]["alternatives"]:              # alternatives in the same normal form as the value
        if is_numeric_type(col_spec) or typ == "qty_text":
            n = to_num(a["value"])
            if isinstance(n, (int, float)):
                a["value"] = n
    return cell

def num(cell):
    v = (cell or {}).get("value")
    return v if isinstance(v, (int, float)) and not isinstance(v, bool) else None

def finalize_cell(c):
    """confidence.final + status from the token confidence, the checks and the agreement of two reads."""
    conf = c["confidence"]
    if c["status"] in ("empty", "no_count"):
        conf["final"] = None
        if c["status"] == "empty" and "fail" in c["checks"].values():    # a required value is missing
            c["status"] = "low"
        return c
    final = conf["model"]
    results = set(c["checks"].values())
    if "pass" in results:
        final = max(final or 0.0, CHECK_PASS_FLOOR)
    if "fail" in results:
        final = min(final if final is not None else CHECK_FAIL_CAP, CHECK_FAIL_CAP)
    if conf["agreement"] == 0:
        final = min(final if final is not None else DISAGREE_CAP, DISAGREE_CAP)
    if c["status"] == "repaired":
        final = min(final if final is not None else REPAIRED_CAP, REPAIRED_CAP)
    conf["final"] = None if final is None else round(final, 3)
    if c["status"] in ("repaired", "low"):
        return c
    if final is None:
        c["status"] = "review"                         # no token scores (dry run / unparsable answer)
    else:
        c["status"] = "ok" if final >= CONF_OK else "review" if final >= CONF_REVIEW else "low"
    return c

def mark(cells, rule, result):
    for c in cells:
        if c is not None:
            if c["checks"].get(rule) != "fail":                 # a failure is never overwritten by a pass
                c["checks"][rule] = result

# ---- restricted expressions for spec rules ----------------------------------------------------------
_ALLOWED = (ast.Expression, ast.Compare, ast.BinOp, ast.UnaryOp, ast.BoolOp, ast.Name, ast.Load, ast.Constant,
            ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Mod, ast.Eq, ast.NotEq, ast.Lt, ast.LtE, ast.Gt, ast.GtE,
            ast.And, ast.Or, ast.Not, ast.USub, ast.Call)
_FUNCS = {"abs": abs, "min": min, "max": max, "round": round}

def rule_names(expr):
    tree = ast.parse(expr, mode="eval")
    return sorted({n.id for n in ast.walk(tree) if isinstance(n, ast.Name) and n.id not in _FUNCS})

def safe_eval(expr, env):
    """-> True / False, or None when a name is missing or the expression cannot be evaluated (division by zero)."""
    tree = ast.parse(expr, mode="eval")
    for n in ast.walk(tree):
        if not isinstance(n, _ALLOWED) or (isinstance(n, ast.Call) and not (isinstance(n.func, ast.Name) and n.func.id in _FUNCS)):
            raise ValueError(f"not allowed in a rule: {type(n).__name__}")
    if any(env.get(k) is None for k in rule_names(expr)):
        return None
    try:
        return bool(eval(compile(tree, "<rule>", "eval"), {"__builtins__": {}, **_FUNCS}, env))
    except (ZeroDivisionError, TypeError, ValueError):
        return None

def rule_sides(expr, env):
    """expected / actual for the report: the two sides of a comparison."""
    try:
        tree = ast.parse(expr, mode="eval").body
        if isinstance(tree, ast.Compare) and len(tree.comparators) == 1:
            g = {"__builtins__": {}, **_FUNCS}
            return (eval(compile(ast.Expression(tree.left), "<l>", "eval"), g, env),
                    eval(compile(ast.Expression(tree.comparators[0]), "<r>", "eval"), g, env))
    except Exception:
        pass
    return None, None

def run_rule(checks, expr, scope, cells_by_name, rule_id=None):
    """Evaluate one spec rule on named cells; record it on the page and on every cell it uses."""
    env = {k: num(c) for k, c in cells_by_name.items()}
    ok = safe_eval(expr, env)
    if ok is None:
        return None
    left, right = rule_sides(expr, env)
    rid = rule_id or slug(expr)[:40]
    checks.append(dict(rule=rid, scope=scope, expression=expr, expected=left, actual=right,
                       result="pass" if ok else "fail"))
    mark([cells_by_name.get(n) for n in rule_names(expr)], rid, "pass" if ok else "fail")
    return ok

## 7. Header, table skeleton and form matching

In [ ]:
HEADER_PROMPT = """This is the top part of a scanned factory form of Meenakshi Polymers.
Transcribe the printed heading and the header fields. Ignore the table rows below the column headers.
Return only JSON:
{"title_lines": [every printed title / heading line, top to bottom, exactly as printed],
 "company": printed company name or null,
 "doc_no": printed DOC NO value or null, "rev_no": REV NO value or null, "rev_date": REV DATE value or null,
 "org_date": ORG DATE / ORIGINAL DATE value or null,
 "month": month (after MONTH) or null, "date": date (after DATE) or null,
 "shift": shift letter (after SHIFT) or null, "part_name": value after PART NAME or null,
 "line": value after LINE / LINE NO or null,
 "report": handwritten text after "DAILY MIS REPORT-" or null,
 "shop": shop or sub-title printed below the title (e.g. "PRESS SHOP", "WELD SHOP", "PENETRATION") or null,
 "period": handwritten period / date range or null,
 "other_fields": [{"label": printed label, "value": value written next to it} for any other header field]}
Copy text exactly as written. Do not guess values that are not on the page (use null)."""
HEADER_KEYS = ["company", "doc_no", "rev_no", "rev_date", "org_date", "month", "date", "shift", "part_name", "line",
               "report", "shop", "period"]

SKELETON_PROMPT = """This is a scanned factory form. Look at its main table.
Return only JSON:
{"column_labels": [the printed column headings of the main table, left to right; for a two-level heading give the
                   lower labels, e.g. "MINOR", "MAJOR" under "FOR REWORK"],
 "row_labels": [the first 12 printed row names of the table (e.g. defect names, part names, parameters), top to
                bottom; [] when the rows are blank lines filled in by hand],
 "rows_printed": true if the rows have printed names, false if each row is a handwritten entry}
Copy the printed text exactly."""

def folder_form(pdf_rel):
    """form_id from the PDF's folder: the deepest entry of FOLDER_FORMS that is a parent folder of the file."""
    parts = [p.lower() for p in Path(pdf_rel).parent.parts]
    best, depth = None, 0
    for folder, fid in FOLDER_FORMS.items():
        f = [p.lower() for p in Path(folder).parts]
        for i in range(len(parts) - len(f) + 1):
            if parts[i:i + len(f)] == f and len(f) > depth:
                best, depth = fid, len(f)
    return best

def override_for(pdf_rel):
    for pat, fid in FORM_OVERRIDE.items():
        if fnmatch.fnmatch(pdf_rel.lower(), pat.lower()) or fnmatch.fnmatch(Path(pdf_rel).name.lower(), pat.lower()):
            return fid
    return None

def label_match(a, b):
    a, b = norm_text(a), norm_text(b)
    if not a or not b:
        return False
    if a == b or (min(len(a), len(b)) >= 4 and (a in b or b in a)):
        return True
    return difflib.SequenceMatcher(None, a, b).ratio() >= 0.8

def columns_f1(read, spec_cols):
    """F1 of read column labels vs the spec's printed labels (recall: spec labels found; precision: read labels
    explained). A spec whose columns are a subset of the page's scores lower than the exact spec."""
    read = [r for r in read if norm_text(r)]
    if not read or not spec_cols:
        return None
    rec = sum(any(label_match(s, r) for r in read) for s in spec_cols) / len(spec_cols)
    prec = sum(any(label_match(r, s) for s in spec_cols) for r in read) / len(read)
    return 0.0 if rec + prec == 0 else round(2 * rec * prec / (rec + prec), 3)

def score_form(spec, header, skeleton, text_layer=""):
    m = spec.get("match") or {}
    h = header if isinstance(header, dict) else {}
    titles = " | ".join(map(str, h.get("title_lines") or [])) + " | " + str(h.get("shop") or "")
    t = max([phrase_score(titles, p) for p in m.get("title_any") or []] or [0.0])
    used_layer = False
    if text_layer and t < 0.5:
        tl = 0.9 * max([phrase_score(text_layer[:3000], p) for p in m.get("title_any") or []] or [0.0])
        if tl > t:
            t, used_layer = tl, True
    excluded = norm_text(titles + (" | " + text_layer[:3000] if used_layer else ""))
    if any(fuzzy_contains(excluded, norm_text(p)) for p in m.get("title_none") or []):
        t *= 0.3
    doc = None
    if m.get("doc_no_any") and h.get("doc_no"):
        d = norm_text(h["doc_no"]).replace(" ", "")
        doc = 1.0 if any(norm_text(x).replace(" ", "") in d for x in m["doc_no_any"]) else 0.0
    read_cols = (skeleton or {}).get("column_labels") if isinstance(skeleton, dict) else None
    cols = columns_f1([str(x) for x in read_cols or []], m.get("columns_any") or [])
    parts = [(0.45, t)] + ([(0.45, cols)] if cols is not None else []) + ([(0.25, doc)] if doc is not None else [])
    score = sum(w * s for w, s in parts) / sum(w for w, _ in parts)
    return round(score, 3), dict(title=round(t, 3), columns=cols, doc_no=doc)

def decide_form(pdf_rel, header, skeleton, text_layer):
    """-> (spec or None, match dict). Order: FORM_OVERRIDE > clear match > folder fallback > generic
    (ambiguous / new_form)."""
    ranked = sorted(((score_form(s, header, skeleton, text_layer), fid) for fid, s in FORMS.items()),
                    key=lambda x: -x[0][0])
    best = ranked[0] if ranked else None
    second = ranked[1] if len(ranked) > 1 else None
    forced, folder = override_for(pdf_rel), folder_form(pdf_rel)
    match = dict(status=None, score=best[0][0] if best else None, decided_by=None,
                 runner_up=dict(form_id=second[1], score=second[0][0]) if second else None,
                 evidence=best[0][1] if best else None, folder=folder, override=forced,
                 candidates=[dict(form_id=f, score=s[0]) for s, f in ranked[:4]], warnings=[])
    def pick(fid, status, how):
        spec = FORMS.get(fid)
        if spec is None:
            match["warnings"].append(f"unknown form id {fid!r}")
            return None, dict(match, status="new_form", decided_by=None)
        sc = next((s for s, f in ranked if f == fid), (None, None))
        return spec, dict(match, status=status, decided_by=how, form_id=fid, score=sc[0], evidence=sc[1])
    if forced:
        return pick(forced, "override", "override")
    if TRUST_FOLDER and folder:
        return pick(folder, "matched", "folder")
    if best and best[0][0] >= MATCH_MIN:
        fam = FORMS[best[1]].get("family")
        same_family = bool(second and fam and fam == FORMS[second[1]].get("family"))
        margin = best[0][0] - (second[0][0] if second else 0)
        # forms of one family (MIS sub-types) differ in a few columns only: when they score close, the folder's form
        # wins if it is one of them; otherwise the best needs a small lead
        near = [f for s, f in ranked if best[0][0] - s[0] < MATCH_MARGIN and FORMS[f].get("family") == fam]
        if same_family and margin < MATCH_MARGIN and folder in near:
            spec, mt = pick(folder, "matched", "page+folder")
            return spec, mt
        if margin >= MATCH_MARGIN or (same_family and margin >= 0.02):
            spec, mt = pick(best[1], "matched", "page")
            if folder and folder != best[1]:
                mt["warnings"].append(f"folder says {folder}, page matches {best[1]}")
            return spec, mt
    if folder:
        fs = next((s[0] for s, f in ranked if f == folder), 0.0)
        if fs >= FOLDER_MIN or DRY_RUN:
            spec, mt = pick(folder, "matched", "folder")
            mt["warnings"].append(f"page match unclear (best {best[1] if best else None} {best[0][0] if best else ''}) "
                                  f"- form taken from the folder")
            return spec, mt
    status = "ambiguous" if best and best[0][0] >= MATCH_MIN else "new_form"
    return None, dict(match, status=status, decided_by="generic", form_id=None)

## 8. Layout kind `record` (supplier reports, line rejection, penetration, scrap note)
Find the handwritten rows, group wrapped lines into one entry and send `[column-header strip + entry]` per entry.
Handwriting that runs out of its cell is given to the entry holding most of it; other entries' writing inside the
crop is painted white. The printed header row comes from the spec (`header_rows`) or, for an unknown form, from the
ruled lines: the band just above the longest run of equally high rows.

In [ ]:
def record_prompt(spec):
    cols = spec["columns"]
    lines = [
        f'This image is cut from a handwritten factory register: "{spec["name"]}" (Meenakshi Polymers, Haridwar).',
        "The top strip shows the printed column headers. Below it is ONE handwritten entry. "
        "The entry may continue onto a second ruled line - treat those lines as the same entry.",
        "Extract the entry into JSON with exactly these keys:"]
    for c in cols:
        typ = "integer" if c.get("type", "int") == "int" else "number" if c.get("type") == "number" else "string"
        lines.append(f'- "{c["key"]}": {typ} or null   (printed column: "{c["labels"][0]}")')
    lines += [
        "Rules:",
        "- Copy exactly what is handwritten. Do NOT calculate, correct, or infer any value.",
        "- Empty or illegible cell -> null. A struck-out value is ignored; use the corrected value.",
        '- Integer fields: digits only, e.g. "07" -> 7, "12,274" -> 12274.',
        '- In text fields, a ditto mark meaning "same as above" (", 〃, or two short strokes like 11) -> "DITTO".',
        "- Keep spelling as written. Join text wrapped onto the next line with a single space.",
        "- Handwriting may run past a column line, past the table border, or slightly above / below the ruled row: "
        "text belongs to the column where it starts.",
        "- Dates: copy as written, e.g. 03/6/26.",
        'Return only JSON in this form: {"entries": [ { ...keys above... } ]}']
    return "\n".join(lines)

def assign_ink(img, ink, spans, x0, x1, pitch, reach_rows, max_side, pads):
    """Handwriting does not respect the ruled cells. Each written word (strokes joined by OVERFLOW_WORD_GAP) is given
    to the span holding most of it; a word straddling two spans is split into strokes, each given to the span holding
    its centre. Each span's crop grows to cover all its writing.
    Returns (crop box per span, [(box, owner)] for the debug image, owner map: span index per ink pixel, -1 = none)."""
    H, W = img.shape[:2]
    owner_map = np.full(ink.shape, -1, np.int16)
    if not spans:
        return [], [], owner_map
    gx, gy = OVERFLOW_WORD_GAP
    n, wlab, wst, _ = cv2.connectedComponentsWithStats(cv2.dilate(ink, cv2.getStructuringElement(cv2.MORPH_RECT, (gx, gy))))
    _, slab, sst, scen = cv2.connectedComponentsWithStats(ink)
    lo_x, hi_x = max(x0 - max_side, 0), min(x1 + max_side, W)
    reach = reach_rows * pitch

    def by_overlap(y0, y1):
        best, best_ov = None, 0
        for k, (t, b) in enumerate(spans):
            ov = min(y1, b) - max(y0, t)
            if ov > best_ov:
                best, best_ov = k, ov
        return best, best_ov

    def by_centre(cy):
        for k, (t, b) in enumerate(spans):
            if t <= cy < b:
                return k
        k = min(range(len(spans)), key=lambda k: min(abs(cy - spans[k][0]), abs(cy - spans[k][1])))
        return k if min(abs(cy - spans[k][0]), abs(cy - spans[k][1])) <= reach else None

    marks = []
    for i in range(1, n):
        x, y, w, h, _ = wst[i]
        if x + w < lo_x or x > hi_x or h > 3 * pitch:                       # margin note / signature / far away
            continue
        if y + h < spans[0][0] - reach or y > spans[-1][1] + reach:
            continue
        best, ov = by_overlap(y, y + h)
        t, b = spans[best] if best is not None else (0, 0)
        if best is not None and ov >= 0.6 * h and y >= t - reach and y + h <= b + reach:
            owner_map[y:y + h, x:x + w][(wlab[y:y + h, x:x + w] == i) & (ink[y:y + h, x:x + w] > 0)] = best
            marks.append(((int(x), int(y), int(x + w), int(y + h)), best))
            continue
        for j in np.unique(slab[y:y + h, x:x + w][wlab[y:y + h, x:x + w] == i]):
            if j == 0:
                continue
            sx, sy, sw, sh, _ = sst[j]
            o = by_centre(scen[j][1])
            if o is not None and sy >= spans[o][0] - reach and sy + sh <= spans[o][1] + reach:
                owner_map[sy:sy + sh, sx:sx + sw][slab[sy:sy + sh, sx:sx + sw] == j] = o
            marks.append(((int(sx), int(sy), int(sx + sw), int(sy + sh)), o))
    side_pad, top_pad, bottom_pad = pads
    boxes = []
    for k, (t, b) in enumerate(spans):
        bx0, by0, bx1, by1 = x0 - side_pad, t - top_pad, x1 + side_pad, b + bottom_pad
        ys, xs = np.where(owner_map == k)
        if len(xs):
            bx0, by0 = min(bx0, xs.min() - 10), min(by0, ys.min() - 6)
            bx1, by1 = max(bx1, xs.max() + 10), max(by1, ys.max() + 6)
        boxes.append((max(int(bx0), 0), max(int(by0), 0), min(int(bx1), W), min(int(by1), H)))
    return boxes, marks, owner_map

def isolated_crop(img, box, owner_map, k):
    """Crop a box and paint white the handwriting of every other span (printed lines stay)."""
    bx0, by0, bx1, by1 = box
    body = img[by0:by1, bx0:bx1].copy()
    om = owner_map[by0:by1, bx0:bx1]
    other = ((om >= 0) & (om != k)).astype(np.uint8)
    body[cv2.dilate(other, np.ones((5, 5), np.uint8)) > 0] = 255
    return body

def auto_header_rows(rows):
    """Index of the first data row line: the longest run of equally spaced row lines is the data area."""
    pitches = list(np.diff(rows))
    run = uniform_run(pitches, tol=0.2, min_len=4)
    if run is None or run[0] == 0:
        return None
    return run[0]

def extract_record(ctx, img, raw_gray, spec):
    """-> (table rows [(answer, conf, prefix, box, crop path)], warnings)."""
    dbg, warnings, lay = ctx.dbg, [], spec.get("layout") or {}
    gray, bw, h, v, rows, cands = basic_grid(img)
    dbg.save("binary", bw)
    footer = lay.get("footer_rows", 1)
    hr = lay.get("header_rows") or auto_header_rows(rows)
    if hr is None:
        hr = 2
        warnings.append("header row not found from the ruled lines - assuming 2 printed rows above the data")
    if len(rows) < hr + footer + 2 or len(cands) < 2:
        auto = auto_header_rows(rows)
        if auto and len(rows) >= auto + footer + 2 and len(cands) >= 2:
            warnings.append(f"spec header_rows {hr} does not fit the {len(rows)} ruled lines - using {auto}")
            hr = auto
        else:
            raise RuntimeError(f"table grid not found (rows={len(rows)}, column lines={len(cands)})")
    cols, col_report = filter_columns(raw_gray, rows, cands)
    if len(cols) < 2:
        raise RuntimeError("no column lines left after filtering")
    if lay.get("expected_col_lines") and len(cols) != lay["expected_col_lines"]:
        warnings.append(f"found {len(cols)} column lines, form expects {lay['expected_col_lines']}")
    ink = ink_mask(gray, h, v);                   dbg.save("ink_no_lines", ink)
    x0, x1 = cols[0], cols[-1]
    data_rows = range(hr, len(rows) - 1 - footer)
    row_ink = {i: int(ink[rows[i] + 4:rows[i + 1] - 4, x0:x1].sum() / 255) for i in data_rows}
    filled = [i for i, s in row_ink.items() if s > MIN_ROW_INK]

    def first_col_ink(i):
        return ink[rows[i] + 4:rows[i + 1] - 4, cols[0] + 4:cols[1] - 4].sum() / 255
    records = []
    for i in filled:
        if records and i == records[-1][-1] + 1 and first_col_ink(i) < CONTINUATION_MAX_INK:
            records[-1].append(i)
        else:
            records.append([i])
    if not records:
        warnings.append("no handwritten rows found")

    ov = img.copy()       # rows red, kept columns blue, dropped columns yellow, header magenta, records green
    for i, y in enumerate(rows):
        cv2.line(ov, (0, y), (ov.shape[1], y), (0, 0, 255), 2)
        cv2.putText(ov, f"r{i}", (5, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    for c in col_report:
        cv2.line(ov, (c["x"], 0), (c["x"], ov.shape[0]), (255, 0, 0) if c["kept"] else (0, 200, 255), 2 if c["kept"] else 5)
    cv2.rectangle(ov, (x0, rows[hr - 1]), (x1, rows[hr]), (255, 0, 255), 5)
    for k, rec in enumerate(records):
        cv2.rectangle(ov, (x0, rows[rec[0]]), (x1, rows[rec[-1] + 1]), (0, 170, 0), 6)
        cv2.putText(ov, f"entry {k}", (x0 + 10, rows[rec[0]] + 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 170, 0), 3)
    ctx.overlay = dbg.save("grid_overlay", ov)
    if SHOW_INLINE:
        show(ov[:min(ov.shape[0], rows[-1] + 100)], f"{ctx.pdf_rel} p{ctx.page_no} – rows red · columns blue "
             "(yellow = dropped shadow line) · header magenta · entries green", width=12)

    pitch = float(np.median(np.diff(rows))) if len(rows) > 1 else 60.0
    spans = [(rows[r[0]], rows[r[-1] + 1]) for r in records]
    boxes, marks, owner_map = assign_ink(img, ink, spans, x0, x1, pitch, OVERFLOW_MAX_ROWS, OVERFLOW_MAX_SIDE,
                                         (SIDE_PAD, TOP_PAD, BOTTOM_PAD))
    ov2 = img.copy()
    palette = [(0, 170, 0), (0, 120, 255), (200, 0, 200), (0, 160, 160), (200, 120, 0)]
    for w, o in marks:
        cv2.rectangle(ov2, (w[0], w[1]), (w[2], w[3]), palette[o % len(palette)] if o is not None else (150, 150, 150), 2)
    for k, (bx0, by0, bx1, by1) in enumerate(boxes):
        c = palette[k % len(palette)]
        cv2.rectangle(ov2, (bx0, by0), (bx1, by1), c, 5)
        cv2.putText(ov2, f"crop {k}", (bx1 - 140, by0 + 34), cv2.FONT_HERSHEY_SIMPLEX, 1.0, c, 3)
    dbg.save("entry_crops", ov2)

    prompt, out = record_prompt(spec), []
    for k, rec in enumerate(records):
        bx0, by0, bx1, by1 = boxes[k]
        body = isolated_crop(img, boxes[k], owner_map, k)
        header = img[rows[hr - 1]:rows[hr], bx0:bx1]
        parsed, call = call_model(ctx, f"entry_{k:02d}", with_margin(np.vstack([header, body])), prompt, 1024)
        got = (parsed or {}).get("entries") if isinstance(parsed, dict) else None
        if call["error"] and not DRY_RUN:
            warnings.append(f"entry {k}: {call['error']}")
        if got is not None and len(got) != 1:
            warnings.append(f"entry {k}: model returned {len(got)} entries for one crop")
        ans = got[0] if got and isinstance(got[0], dict) else {}
        out.append(dict(answer=ans, conf=call["conf"], prefix=("entries", 0), bbox=list(boxes[k]),
                        crop=call["image"], ruled_rows=rec))
    return out, warnings

def record_table(spec, entries):
    """Entries -> one table (cells, ditto resolved, required / pattern / rule checks applied later)."""
    cols = spec["columns"]
    rows, prev = [], {}
    for k, e in enumerate(entries):
        cells = {}
        for c in cols:
            raw = e["answer"].get(c["key"]) if isinstance(e["answer"], dict) else None
            cell = make_cell(raw, c, e["conf"].get(e["prefix"] + (c["key"],)))
            if isinstance(raw, str) and raw.strip().upper() == "DITTO" and c["key"] in prev:
                cell.update(value=prev[c["key"]], source="ditto", status=None)
            cells[c["key"]] = cell
        prev.update({key: c["value"] for key, c in cells.items() if c["value"] is not None})
        rows.append(dict(row_idx=k + 1, row_type="entry", section=None, labels={}, cells=cells,
                         bbox=e["bbox"], crop=e["crop"]))
    return [dict(table_no=1, kind="record", block=None, columns=[column_meta(c) for c in cols], rows=rows)]

def column_meta(c, role=None):
    return dict(key=c["key"], label=c.get("label") or c["labels"][0], group=c.get("group"),
                role=role or c.get("role") or "value", type=c.get("type", "int"),
                cell_format=c.get("cell_format", "plain"))

def check_record_table(t, spec, checks):
    for r in t["rows"]:
        scope = f"entry {r['row_idx']}"
        for c in spec["columns"]:
            cell = r["cells"][c["key"]]
            if c.get("required") and cell["value"] is None:
                mark([cell], "required", "fail")
                checks.append(dict(rule="required", scope=f"{scope} {c['key']}", expression=f"{c['key']} is filled",
                                   expected=None, actual=None, result="fail"))
            if cell["checks"].get("pattern") == "fail":
                checks.append(dict(rule="pattern", scope=f"{scope} {c['key']}", expression=c["pattern"],
                                   expected=None, actual=cell["value"], result="fail"))
        for expr in spec.get("rules") or []:
            run_rule(checks, expr, scope, r["cells"])

## 9. Layout kind `matrix` (MIS sheets, QG-2 weld checksheets, weld shop rework & rejection sheet)
1. **Locate** the column-header block (Qwen on the half-size page → Qwen on the top 40 % → ruled lines). Each
   candidate is **verified by reading its column names** (a row-name column from `header_terms`, and a TOTAL column
   unless `require_total` is false). None passing = the page fails for review before any row is read.
2. **Columns**: the header strip is read as leaf columns + group heading; labels are aligned to the spec's columns
   (keys, roles, types, `tally_with_count`). Unknown form: keys from the printed labels.
3. **Rows** from the PRINTED row names (the strip between column lines `row_strip_cols`). Each model input =
   `[header strip + ONE row]`; other rows' handwriting is painted white. `ROW_BATCH` rows share one GPU call.
4. **Assemble**: data rows (by SR.NO, or by row name for fixed-row sheets), section headings, section totals,
   % rows, summary rows, and blocks (the rework sheet repeats its header for shift B).

In [ ]:
MIS_LOCATE_PROMPT = """This is a scanned factory sheet with a large table.
Locate:
1. "header": the table's printed column-header block - the row(s) holding the column names (such as SR.NO, DEFECT,
   DESCRIPTION, part names, time slots, TOTAL), including a group heading above some columns (e.g. "FOR REWORK",
   "OBSERVATION"). Not the company title.
2. "labels": the printed column holding the row names (DEFECT / DESCRIPTION / PART NAME), from below the column
   header down to the last row of the table.
3. "table": the whole table from the column-header block down to its last printed row (summary rows such as
   TOTAL PRODUCTION included), not the signatures below it.
Return only JSON: {"header": [x1, y1, x2, y2], "labels": [x1, y1, x2, y2], "table": [x1, y1, x2, y2]} with
coordinates on a 0-1000 scale relative to the image width and height."""

MIS_LOCATE_HEADER_PROMPT = """This is the top part of a scanned factory sheet with a table.
Locate the table's printed column-header block: the row(s) holding the column names (such as SR.NO, DEFECT,
DESCRIPTION, part names, time slots, TOTAL), including a group heading above some columns. Not the company title,
not the first data row.
Return only JSON: {"header": [x1, y1, x2, y2]} with coordinates on a 0-1000 scale relative to the image width and height."""

MIS_COLUMNS_PROMPT = """This is the printed column-header block of a factory table.
Some columns have a two-level heading: a group heading on top spans several columns (for example "FOR REWORK" above
"MINOR" and "MAJOR", or "OBSERVATION" above "7 TO 9", "9 TO 11"), while other columns (for example "TOTAL") span both
levels. List the bottom-level columns from left to right - one entry for every column in which values are written,
including SR.NO / DEFECT / DESCRIPTION / PART NAME. For each give the printed label (join the words of a label printed
on several lines with a space) and the group heading printed above it, or null when there is none.
Return only JSON: {"columns": [{"label": "SR.NO", "group": null}, {"label": "MINOR", "group": "FOR REWORK"}]}"""

BOX_NAMES = {"header": lambda l: "header" in l,
             "labels": lambda l: "label" in l or "defect" in l or "sr" in l.split(),
             "table": lambda l: "table" in l and "header" not in l}

def get_box(parsed, name):
    """Accepts {"header": [..]}, {"header": {"bbox_2d": [..]}} and Qwen's grounding style
    [{"bbox_2d": [..], "label": "header"}, ...]."""
    if isinstance(parsed, dict):
        v = parsed.get(name)
        if isinstance(v, dict):
            v = v.get("bbox_2d") or v.get("bbox")
        if v is None and isinstance(parsed.get("bbox_2d"), list) and BOX_NAMES[name](str(parsed.get("label", "")).lower()):
            v = parsed["bbox_2d"]
        return v
    if isinstance(parsed, list):
        for o in parsed:
            if isinstance(o, dict) and BOX_NAMES[name](str(o.get("label", "")).lower()):
                return o.get("bbox_2d") or o.get("bbox")
    return None

def box_px(b, W, H, view_wh):
    """[x1,y1,x2,y2] from the model -> page pixels (0-1000 scale; absolute model-view pixels accepted too)."""
    try:
        b = [float(x) for x in b][:4]
    except (TypeError, ValueError):
        return None
    if len(b) != 4:
        return None
    sx, sy = (W / view_wh[0], H / view_wh[1]) if max(b) > 1000 and view_wh else (W / 1000, H / 1000)
    x1, y1, x2, y2 = b[0] * sx, b[1] * sy, b[2] * sx, b[3] * sy
    return [int(min(x1, x2)), int(min(y1, y2)), int(max(x1, x2)), int(max(y1, y2))]

def header_plausible(hb, H):
    return hb is not None and 0.008 * H <= hb[3] - hb[1] <= 0.15 * H and hb[3] < 0.5 * H

def complete_layout(lay, W, cands, lines):
    """Fill table sides / bottom from ruled lines when the model gave only the header."""
    if lay.get("x") is None:
        lay["x"] = ((max(min(cands) - 25, 0), min(max(cands) + 25, W)) if len(cands) >= 2
                    else (int(0.03 * W), int(0.97 * W)))
    if lay.get("bottom") is None:
        long_lines = [q["y"] for q in lines if q["x1"] - q["x0"] > 0.45 * W and q["y"] > lay["header_y"][1]]
        lay["bottom"] = max(long_lines) + 15 if long_lines else None
    lay["row_lines"] = [q["y"] for q in lines if q["x1"] - q["x0"] > 0.3 * (lay["x"][1] - lay["x"][0])]
    return lay

def matrix_layout_candidates(ctx, img, cands, notes):
    """Possible header / table locations, best first (generator: a model call only when the previous one failed)."""
    H, W = img.shape[:2]
    lines = find_line_pieces(horizontal_lines(binarize(to_gray(img))))
    seen = []
    tb, table_ok, lb = None, False, None
    if not DRY_RUN:
        small = cv2.resize(img, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
        parsed, call = call_model(ctx, "mis_locate", small, MIS_LOCATE_PROMPT, 300)
        hb = box_px(get_box(parsed, "header"), W, H, call.get("model_view_wh"))
        tb = box_px(get_box(parsed, "table"), W, H, call.get("model_view_wh"))
        lb = box_px(get_box(parsed, "labels"), W, H, call.get("model_view_wh"))
        table_ok = tb is not None and tb[2] - tb[0] > 0.5 * W
        if header_plausible(hb, H):
            seen.append(hb[3])
            yield complete_layout(dict(header_y=(max(hb[1] - 6, 0), hb[3] + 4),
                                       x=(max(tb[0] - 25, 0), min(tb[2] + 25, W)) if table_ok else None,
                                       bottom=min(tb[3] + 15, H) if table_ok and tb[3] > hb[3] + 0.2 * H else None,
                                       labels_box=lb, method="model"), W, cands, lines)
        else:
            notes.append(f"locate: header box missing/implausible ({hb}); raw: {call['raw'][:150]!r}")
        top = img[:int(0.4 * H)]
        parsed2, call2 = call_model(ctx, "mis_locate_header", top, MIS_LOCATE_HEADER_PROMPT, 120)
        hb2 = box_px(get_box(parsed2, "header"), W, top.shape[0], call2.get("model_view_wh"))
        if header_plausible(hb2, H) and all(abs(hb2[3] - y) > 15 for y in seen):
            seen.append(hb2[3])
            yield complete_layout(dict(header_y=(max(hb2[1] - 6, 0), hb2[3] + 4),
                                       x=(max(tb[0] - 25, 0), min(tb[2] + 25, W)) if table_ok else None,
                                       bottom=None, labels_box=lb, method="model_top_crop"), W, cands, lines)
        elif not header_plausible(hb2, H):
            notes.append(f"locate (top crop): header box missing/implausible ({hb2})")
    # ruled lines: column lines that reach the table bottom start at the header top; the next long row line ends it
    vm = vertical_lines(binarize(to_gray(img)))
    tops = []
    for x in cands:
        ys = np.where(vm[:, max(x - 3, 0):x + 4].max(axis=1) > 0)[0]
        if len(ys) > 0.3 * H:
            tops.append(int(ys.min()))
    long_lines = [q for q in lines if q["x1"] - q["x0"] > 0.45 * W]
    if len(tops) >= 3 and long_lines:
        y_top = int(np.median(tops))
        below = [q["y"] for q in long_lines if q["y"] > y_top + 28]
        if below and all(abs(below[0] - y) > 15 for y in seen):
            yield complete_layout(dict(header_y=(y_top, below[0]), x=None, bottom=None, labels_box=None,
                                       method="ruled_lines"), W, cands, lines)

NAME_COL_TERMS = ["SR NO", "S NO", "SR N", "SNO", "DEFECT", "DESCRIPTION"]

def header_check(read, terms=None, require_total=True):
    """A real column header has a row-name column and (usually) a TOTAL column and is not made of numbers."""
    labels = [norm_text(c.get("label")) for c in read or [] if isinstance(c, dict) and c.get("label")]
    if len(labels) < 4:
        return False, f"only {len(labels)} column names read"
    names = any(fuzzy_contains(l, norm_text(t), 0.8) for l in labels[:4] for t in terms or NAME_COL_TERMS)
    numeric = sum(bool(re.fullmatch(r"[\d\s./%-]+", l)) for l in labels)
    if not names:
        return False, f"no row-name column ({', '.join((terms or NAME_COL_TERMS)[:3])} …) among {labels[:4]}"
    if require_total and not any(fuzzy_contains(l, "TOTAL", 0.8) for l in labels):
        return False, "no TOTAL column"
    if numeric > 1 and not any(re.search(r"\bTO\b", l) for l in labels):
        return False, f"{numeric} column names are numbers (a data row, not the header)"
    return True, "ok"

def row_strip_x(layout, cands, n_label, strip_cols=None):
    """x ranges of the printed row names: between column lines (spec `row_strip_cols`: one [a, b] pair of column-line
    indexes, or several pairs whose rows are merged - e.g. S.No + DESCRIPTION, where summary rows are labelled in the
    S.No column); default from the table's left line to the line after the label columns. Then the model's box; a
    fixed 15 % only as the last resort. -> ([(x0, x1)], how)"""
    xa, xb = layout["x"]
    tw = xb - xa
    lines = sorted(x for x in cands if xa - 5 <= x <= xb + 5)
    pairs = ([tuple(p) for p in strip_cols] if strip_cols and isinstance(strip_cols[0], (list, tuple))
             else [tuple(strip_cols)] if strip_cols else [(0, n_label)])
    widths = np.diff(lines) if len(lines) > 1 else np.array([])
    ranges = []
    for a, b in pairs:
        if len(lines) <= b:
            break
        left, right = lines[a], lines[b]
        ok_default = (strip_cols is not None or (abs(left - (xa + 25)) < 0.05 * tw and len(widths) > n_label
                                                  and max(widths[:n_label]) > np.median(widths)))
        if 0.015 * tw <= right - left <= 0.45 * tw and ok_default:
            ranges.append((left + 4, right - 4))
    if ranges and len(ranges) == len(pairs):
        return ranges, "column lines"
    lb = layout.get("labels_box")
    if lb is not None and 20 < lb[2] - lb[0] < 0.5 * tw:
        return [(max(lb[0] - 10, 0), lb[2] + 10)], "model box"
    return [(xa + 15, xa + int(0.15 * tw))], "fixed 15 % of the table (not measured - check the overlay)"

def printed_text_bands(img, x0, x1, y0, y1):
    """Vertical bands holding printed text in one strip (grid lines removed, letter-sized blobs only)."""
    g = cv2.GaussianBlur(to_gray(img)[y0:y1, x0:x1], (5, 5), 0)
    ink = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 31, 18)
    hl = cv2.morphologyEx(ink, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (60, 1)))
    vl = cv2.morphologyEx(ink, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, 45)))
    ink = cv2.subtract(ink, cv2.dilate(hl | vl, np.ones((5, 5), np.uint8)))
    n, lab, st, _ = cv2.connectedComponentsWithStats(ink)
    ok = np.zeros(n, bool)
    ok[1:] = (st[1:, 3] >= 8) & (st[1:, 3] <= 60) & (st[1:, 4] >= 25) & (st[1:, 2] <= 260)
    on = ok[lab].sum(axis=1) >= 3
    bands, start = [], None
    for y, val in enumerate(list(on) + [False]):
        if val and start is None:
            start = y
        elif not val and start is not None:
            if y - start >= 6:
                bands.append([start + y0, y + y0])
            start = None
    return bands

def matrix_text_rows(img, layout, allow_gaps=False):
    """Rows from the PRINTED row names: that strip has no handwriting, and printed text stays detectable on faint
    photocopies where the ruled lines break up. Several strips: their rows are merged.
    Returns ([(top, bottom)], pitch, filled)."""
    H = img.shape[0]
    y0 = layout["header_y"][1] + 3
    y1 = layout["bottom"] or int(0.97 * H)
    bands = sorted(b for lx0, lx1 in layout["labels_x"] for b in printed_text_bands(img, lx0, lx1, y0, y1))
    if len(bands) < 3:
        return [], None, 0
    centers = sorted(set((a + b) / 2 for a, b in bands))
    pitch = float(np.median([d for d in np.diff(centers) if d > 8] or [40]))
    merged = [list(bands[0])]
    for a, b in bands[1:]:                       # 2nd line of a two-line name / the same row seen in two strips
        if (a + b) / 2 - (merged[-1][0] + merged[-1][1]) / 2 < 0.6 * pitch:
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a, b])
    centers = [(a + b) / 2 for a, b in merged]
    d = np.diff(centers)
    sel = d[(d > 0.5 * pitch) & (d < 2 * pitch)] if len(d) else d
    pitch = float(np.median(sel)) if len(sel) else pitch
    if layout["bottom"] is None and not allow_gaps:   # table end unknown: stop at the first big gap in the lower half
        for i in range(len(centers) - 1):
            if centers[i + 1] - centers[i] > 3.5 * pitch and centers[i] > 0.5 * H:
                centers = centers[:i + 1]
                break
    filled, out = 0, [centers[0]]
    for c in centers[1:]:                        # a row whose name was not detected: fill by the row spacing
        k = int(round((c - out[-1]) / pitch)) - 1
        if 1 <= k <= 4:
            out += [out[-1] + (c - out[-1]) * (j + 1) / (k + 1) for j in range(k)]
            filled += k
        out.append(c)
    spans = []
    for i, c in enumerate(out):
        top_ = (out[i - 1] + c) / 2 if i else max(y0, c - pitch / 2)
        bot = (c + out[i + 1]) / 2 if i + 1 < len(out) else min(y1, c + pitch / 2)
        spans.append((int(top_), int(bot)))
    return spans, pitch, filled

def align_columns(read, canon):
    """Needleman-Wunsch style alignment of read labels to the spec's columns -> [(read_idx|None, canon_idx|None)]."""
    def text(c): return norm_text(f"{c.get('group') or ''} {c.get('label') or ''}")
    def canon_texts(c): return [norm_text(f"{c.get('group') or ''} {l}") for l in c["labels"]] + [norm_text(l) for l in c["labels"]]
    n, m = len(read), len(canon)
    sim = [[max(difflib.SequenceMatcher(None, text(read[i]), t).ratio() for t in canon_texts(canon[j]))
            for j in range(m)] for i in range(n)]
    S = np.zeros((n + 1, m + 1)); B = np.zeros((n + 1, m + 1), dtype=int)
    for i in range(1, n + 1): S[i][0], B[i][0] = S[i - 1][0] - 0.3, 1
    for j in range(1, m + 1): S[0][j], B[0][j] = S[0][j - 1] - 0.3, 2
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            opts = [S[i - 1][j - 1] + (sim[i - 1][j - 1] if sim[i - 1][j - 1] >= 0.45 else -1), S[i - 1][j] - 0.3, S[i][j - 1] - 0.3]
            B[i][j] = int(np.argmax(opts)); S[i][j] = max(opts)
    pairs, i, j = [], n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and B[i][j] == 0:
            pairs.append((i - 1, j - 1)); i -= 1; j -= 1
        elif i > 0 and (j == 0 or B[i][j] == 1):
            pairs.append((i - 1, None)); i -= 1
        else:
            pairs.append((None, j - 1)); j -= 1
    return pairs[::-1]

LABEL_KEYS = {"sr_no": ["SR NO", "S NO", "SR N", "SNO", "SR"], "defect": ["DEFECT", "DESCRIPTION OF DEFECT", "DESCRIPTION"],
              "reason": ["REASON"]}

def guess_roles(cols):
    """Unknown form / unaligned columns: label columns first, value columns before TOTAL are parts, after it dispositions."""
    seen_total = False
    for c in cols:
        if c.get("role"):
            seen_total = seen_total or c["role"] == "total"
            continue
        if c["key"] in LABEL_KEYS:
            c["role"] = "label"
        elif seen_total and norm_text(c["label"]) == "TOTAL":
            c["role"], c["key"] = "grand_total", "grand_total"
        elif norm_text(c["label"]) == "TOTAL":
            c["role"], seen_total = "total", True
        else:
            c["role"] = "disposition" if seen_total else "part"
    keys = set()
    for c in cols:
        k, n = c["key"], 2
        while c["key"] in keys:
            c["key"] = f"{k}_{n}"; n += 1
        keys.add(c["key"])
    return cols

def matrix_columns(read, spec):
    """read: [{label, group}] from the page -> (columns, warnings). Spec columns give key / role / type / format."""
    warnings = []
    read = [dict(label=str(c.get("label") or "").strip(), group=(str(c["group"]).strip() if c.get("group") else None))
            for c in (read or []) if isinstance(c, dict) and str(c.get("label") or "").strip()]
    canon = [dict(c, label=c["labels"][0]) for c in (spec or {}).get("columns") or []]
    if not read:
        if canon:
            warnings.append("columns not read from the page - using the form's column list")
            return guess_roles([dict(c) for c in canon]), warnings
        return [], ["columns not read and no form known"]
    if not canon:
        cols = []
        for r in read:
            n = norm_text(r["label"])
            key = next((k for k, alts in LABEL_KEYS.items() if n in alts or any(n.startswith(a) for a in alts[:2])), None)
            cols.append(dict(label=r["label"], labels=[r["label"]], group=r["group"], type="number",
                             key=key or slug(f"{r['group']} {r['label']}" if r["group"] else r["label"])))
        return guess_roles(cols), warnings
    cols = []
    for ri, ci in align_columns(read, canon):
        if ri is None:
            warnings.append(f"form column {canon[ci]['label']!r} not found on the page")
            continue
        r = read[ri]
        if ci is None:
            warnings.append(f"column {r['label']!r} (group {r['group']}) is not in form {spec['form_id']}")
            cols.append(dict(label=r["label"], labels=[r["label"]], group=r["group"], type="number",
                             key=slug(f"{r['group']} {r['label']}" if r["group"] else r["label"])))
        else:
            cols.append(dict(canon[ci], label=r["label"], group=r["group"] or canon[ci].get("group")))
    return guess_roles(cols), warnings

def row_name_key(cols, spec):
    want = ((spec or {}).get("layout") or {}).get("row_name_key")
    labels = [c["key"] for c in cols if c["role"] == "label"]
    if want in labels:
        return want
    return next((k for k in labels if k != "sr_no"), labels[0] if labels else None)

def matrix_row_prompt(cols, spec):
    name_key = row_name_key(cols, spec)
    title = (spec or {}).get("name") or "factory table"
    lines = [
        f'This image is cut from a handwritten factory sheet: "{title}" (Meenakshi Polymers).',
        "The top strip is the printed column header. Some columns sit under a group heading. Below it is ONE printed "
        "row of the table; thin slices of the rows above and below may show at the edges - ignore them.",
        "Return this one row as JSON with these keys:"]
    if any(c["key"] == "sr_no" for c in cols):
        lines.append('- "sr_no": printed serial number (integer) or null')
    name_col = next((c for c in cols if c["key"] == name_key), None)
    lines.append(f'- "label": the printed row name in column "{name_col["label"] if name_col else "DESCRIPTION"}", '
                 'e.g. "PEEL OFF", "TOTAL PRODUCTION"')
    tally = False
    for c in cols:
        if c["role"] == "label" and c["key"] not in ("sr_no", name_key):
            lines.append(f'- "{c["key"]}": printed text in column "{c["label"]}" or null')
    for c in cols:
        if c["role"] in ("label", "ignore"):
            continue
        g = f' under "{c["group"]}"' if c.get("group") else ""
        if c.get("cell_format") == "tally_with_count":
            tally = True
            what = 'written count (integer), "NC" or null'
        elif c.get("type") == "qty_text":
            what = "what is written (a number, or text such as \"12 Bolt welded\") or null"
        elif c.get("type") in ("text", "date"):
            what = "handwritten text or null"
        else:
            what = "number or null"
        lines.append(f'- "{c["key"]}": {what}   (column "{c["label"]}"{g})')
    lines += [
        "Rules:",
        "- Keys are listed left to right in the same order as the columns on the sheet. Use the column header above "
        "each value to decide its key.",
        "- Copy exactly what is written. Do NOT calculate, total or correct anything.",
        "- A dash (-, —, ~) written in a cell = 0. An empty cell = null. Percent values: the number without %.",
        "- A struck-out value is ignored; use the corrected value."]
    if tally:
        lines += [
            "- Count cells are written as tally strokes followed by the count, e.g. \"|| — ②\" or \"||| - 3\": return "
            "ONLY the number written after the dash or inside the circle (2, 3). NEVER count the strokes yourself.",
            '- Strokes with no written number -> "NC". Empty cell or only a dash -> null.']
    lines.append("Return only the JSON object.")
    return "\n".join(lines)

def unwrap_row(parsed):
    """-> (row dict, json path prefix of its values)."""
    if isinstance(parsed, dict) and isinstance(parsed.get("rows"), list):
        return (parsed["rows"][0], ("rows", 0)) if parsed["rows"] and isinstance(parsed["rows"][0], dict) else (None, ())
    if isinstance(parsed, list):
        return (parsed[0], (0,)) if parsed and isinstance(parsed[0], dict) else (None, ())
    return (parsed, ()) if isinstance(parsed, dict) else (None, ())

def looks_like_neighbour(a, b):
    if not a or not b:
        return False
    sa, sb = to_int(a.get("sr_no")), to_int(b.get("sr_no"))
    if sa is not None and sb is not None:
        return sa == sb
    la, lb = norm_text(a.get("label")), norm_text(b.get("label"))
    return bool(la) and bool(lb) and difflib.SequenceMatcher(None, la, lb).ratio() >= 0.9

def extract_matrix(ctx, img, spec):
    lay_spec = (spec or {}).get("layout") or {}
    warnings, H = [], img.shape[0]
    gray, bw, h, v, _, cands = basic_grid(img)

    # 1. layout: first candidate whose header strip reads as a real column header
    layout, read, rejected, notes = None, None, [], []
    for cand in matrix_layout_candidates(ctx, img, cands, notes):
        (hy0, hy1), (xa, xb) = cand["header_y"], cand["x"]
        if DRY_RUN:
            layout = cand
            warnings.append(f"dry run: header from {cand['method']} not verified by reading its columns")
            break
        parsed, _ = call_model(ctx, f"mis_columns_{cand['method']}", with_margin(img[hy0:hy1, xa:xb]),
                               MIS_COLUMNS_PROMPT, 1024)
        r = (parsed or {}).get("columns") if isinstance(parsed, dict) else None
        ok, why = header_check(r, lay_spec.get("header_terms"), lay_spec.get("require_total", True))
        if ok:
            layout, read = cand, r
            break
        rejected.append(cand)
        warnings.append(f"header candidate from {cand['method']} rejected: {why}")
    warnings += notes
    if layout is None:
        ov = img.copy()
        for c in rejected:
            cv2.rectangle(ov, (c["x"][0], c["header_y"][0]), (c["x"][1], c["header_y"][1]), (0, 0, 255), 5)
        ctx.overlay = ctx.dbg.save("matrix_layout_rejected", ov)
        raise RuntimeError("matrix: no column header found that reads like one - page needs review; "
                           "see the matrix_layout_rejected debug image")
    (hy0, hy1), (xa, xb) = layout["header_y"], layout["x"]

    # 2. columns
    cols, w = matrix_columns(read, spec)
    warnings += w
    if not cols:
        raise RuntimeError("matrix columns unknown (no column header read, no form known)")
    n_label = 0
    for c in cols:
        if c["role"] != "label":
            break
        n_label += 1

    # 3. printed rows
    layout["labels_x"], lx_how = row_strip_x(layout, cands, max(n_label, 1), lay_spec.get("row_strip_cols"))
    if not lx_how.startswith("column"):
        warnings.append(f"row-name strip from {lx_how}")
    spans, pitch, filled = matrix_text_rows(img, layout, lay_spec.get("allow_gaps", False))
    min_rows = lay_spec.get("min_rows", MIS_MIN_ROWS)
    if len(spans) < min_rows:
        raise RuntimeError(f"matrix: only {len(spans)} printed rows found below the header (minimum {min_rows}) "
                           "- page needs review; see the matrix_layout debug image")
    if filled:
        warnings.append(f"{filled} row(s) placed by row spacing (row name not detected) - check the overlay")
    layout["bottom"] = layout["bottom"] or spans[-1][1] + 10
    bottom = layout["bottom"]

    # 4. one crop per row: own handwriting only
    ink = ink_mask(gray, h, v)
    boxes, marks, owner_map = assign_ink(img, ink, spans, xa, xb, pitch, MIS_ROW_REACH, 120, (0, 2, 2))
    ov = img.copy()
    cv2.rectangle(ov, (xa, hy0), (xb, hy1), (255, 0, 255), 6)
    cv2.putText(ov, f"header ({layout['method']})", (xa + 10, hy0 - 10), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 0, 255), 3)
    cv2.rectangle(ov, (xa, hy0), (xb, bottom), (255, 0, 0), 3)
    for sx0, sx1 in layout["labels_x"]:
        cv2.rectangle(ov, (sx0, hy1), (sx1, bottom), (200, 200, 0), 3)
    lx1 = max(x for _, x in layout["labels_x"])
    for k, (bx0, b0, bx1, b1) in enumerate(boxes):
        c = (0, 170, 0) if k % 2 == 0 else (0, 120, 255)
        cv2.rectangle(ov, (lx1 + 6 + 6 * (k % 2), b0 + 2), (bx1 - 6 - 6 * (k % 2), b1 - 2), c, 3)
        cv2.putText(ov, str(k), (bx1 - 70, b1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.9, c, 2)
    ctx.overlay = ctx.dbg.save("matrix_layout", ov)
    if SHOW_INLINE:
        show(ov, f"{ctx.pdf_rel} p{ctx.page_no} – matrix ({layout['method']}): header magenta · row names cyan "
             f"({lx_how}) · {len(boxes)} rows", width=10)

    # 5. read the rows
    prompt = matrix_row_prompt(cols, spec)

    def crop_for(k, tight=False):
        bx0, by0, bx1, by1 = boxes[k]
        if tight:
            by0, by1 = spans[k]
        body = isolated_crop(img, (bx0, by0, bx1, by1), owner_map, k)
        return with_margin(np.vstack([img[hy0:hy1, bx0:bx1], body]))

    def read_rows(ks, tight=False):
        tag = "_retry" if tight else ""
        res = call_model_batch(ctx, [f"row_{k:02d}{tag}" for k in ks], [crop_for(k, tight) for k in ks], prompt, 512)
        out = {}
        for k, (parsed, call) in zip(ks, res):
            ans, prefix = unwrap_row(parsed)
            out[k] = dict(answer=ans, conf=call["conf"], prefix=prefix, bbox=list(boxes[k]), crop=call["image"],
                          error=call["error"])
        return out

    answers = {}
    for i in range(0, len(boxes), max(ROW_BATCH, 1)):
        answers.update(read_rows(list(range(i, min(i + max(ROW_BATCH, 1), len(boxes))))))
    suspects = [k for k in sorted(answers) if any(
        j in answers and looks_like_neighbour(answers[k]["answer"], answers[j]["answer"]) for j in (k - 1, k + 1))]
    if suspects and not DRY_RUN:                     # read from the neighbour's line: re-read with a tight crop
        for k, a in read_rows(suspects, tight=True).items():
            if a["answer"]:
                answers[k] = a
    rows = [dict(answers[k], k=k) for k in sorted(answers) if answers[k]["answer"]]
    if not DRY_RUN and len(rows) < 0.8 * len(boxes):
        warnings.append(f"only {len(rows)} of {len(boxes)} row crops gave an answer")
    ctx.extra = dict(header_method=layout["method"], row_name_strip=lx_how, row_pitch=pitch,
                     rows_placed_by_spacing=filled, rows_reread_neighbour=len(suspects) if not DRY_RUN else 0)
    return dict(cols=cols, rows=rows, reread=lambda ks: read_rows(ks, tight=True)), warnings

# ---- assembling rows into sections / blocks -----------------------------------------------------------
SUMMARY_TERMS = ["OVER ALL", "OVERALL", "PRODUCTION", "REWORK PLANT", "CUSTOMER RETURN", "TOTAL DEFECT QTY",
                 "TOTAL REJECTION", "REVISION"]
FOOTER_TERMS  = ["PREPARED BY", "REVIEWED BY", "APPROVED BY", "CHECKED BY", "REVISION DETAIL", "B GRADE CARRIER"]

def is_total_label(label):  return any_term(label, ["TOTAL"])
def is_pct_label(label):    return "%" in label or any_term(label, ["STRAIGHT PASS", "STRAIGHTPASS", "PPM"])

def section_name(total_label, heading, first_row, other_label_keys):
    if heading:
        return heading
    name = re.sub(r"\b(TOTAL|DEFECTS?|QTY|REJECTION|OF THE DAY)\b", " ", norm_text(total_label or "")).strip()
    if name:
        return re.sub(r"\s+", " ", name)
    for k in other_label_keys:
        v = ((first_row or {}).get("answer") or {}).get(k)
        if v:
            return str(v)
    return None

def assemble_matrix(rows, cols, spec):
    """Row answers in sheet order -> [block {name, items: [(row_type, section, row)]}]."""
    lay = (spec or {}).get("layout") or {}
    row_id = lay.get("row_id", "sr_no")
    blocks_spec = lay.get("blocks") or {}
    split = [norm_text(x) for x in blocks_spec.get("split_labels") or []]
    names = blocks_spec.get("names") or []
    summary_labels = [l for s in (spec or {}).get("summary_rows") or [] for l in s["labels"]]
    footer_terms = FOOTER_TERMS + list(lay.get("footer_terms") or [])
    value_keys = [c["key"] for c in cols if c["role"] not in ("label", "ignore")]
    name_key = row_name_key(cols, spec)
    other_label_keys = [c["key"] for c in cols if c["role"] == "label" and c["key"] not in ("sr_no", name_key)]

    groups = [[]]
    for r in rows:                                        # 1. blocks (a repeated column header starts the next one)
        label = norm_text(r["answer"].get("label"))
        has_vals = any(r["answer"].get(k) not in (None, "") for k in value_keys)
        if split and not has_vals and any(fuzzy_contains(label, s, 0.8) for s in split):
            if groups[-1]:
                groups.append([])
            continue
        groups[-1].append(r)

    out = []
    for gi, grp in enumerate(groups):
        sections, summary, cur, heading, heading_row, in_summary = [], [], None, None, None, False
        for r in grp:
            a = r["answer"]
            label = norm_text(a.get("label"))
            sr = to_int(a.get("sr_no"))
            has_vals = any(a.get(k) not in (None, "") for k in value_keys)
            summary_like = any_term(label, SUMMARY_TERMS, 0.85) or (summary_labels and any_term(label, summary_labels, 0.85))
            footer_like = label.startswith("NOTE") or any_term(label, footer_terms, 0.85)
            if not label and not has_vals and sr is None:
                continue
            if footer_like and not has_vals:
                continue
            is_total = is_total_label(label) and sr is None
            is_pct = is_pct_label(label) and sr is None
            data = (bool(label) and not summary_like and not footer_like) if row_id == "label" else (sr is not None)
            if data and not in_summary:
                restart = (row_id == "sr_no" and cur is not None and sr == 1 and cur["rows"]
                           and (to_int(cur["rows"][-1]["answer"].get("sr_no")) or 0) > 1)
                if cur is None or cur["total"] is not None or heading is not None or restart:
                    cur = dict(rows=[], total=None, pct=None, heading=heading, heading_row=heading_row)
                    sections.append(cur); heading, heading_row = None, None
                cur["rows"].append(r)
            elif is_pct and cur is not None and cur["total"] is not None and cur["pct"] is None and not in_summary:
                cur["pct"] = r
            elif is_total and not is_pct and not summary_like and cur is not None and cur["total"] is None \
                    and cur["rows"] and not in_summary:
                cur["total"] = r
            elif not has_vals and not is_total and not is_pct and label and not summary_like and not in_summary:
                heading, heading_row = a.get("label"), r          # e.g. "PIPE B&C", "FABRICATION DEFECT"
            elif label or has_vals:
                in_summary = in_summary or bool(cur and cur["rows"]) or row_id == "label"
                summary.append(r)
        items = []
        for s in sections:
            name = section_name((s["total"] or {}).get("answer", {}).get("label") if s["total"] else None,
                                s["heading"], (s["rows"] or [None])[0], other_label_keys)
            if s["heading_row"] is not None:
                items.append(("heading", name, s["heading_row"]))
            items += [("data", name, r) for r in s["rows"]]
            if s["total"]:
                items.append(("section_total", name, s["total"]))
            if s["pct"]:
                items.append(("section_pct", name, s["pct"]))
        items += [("summary", None, r) for r in summary]
        out.append(dict(name=names[gi] if gi < len(names) else (f"block {gi + 1}" if len(groups) > 1 else None),
                        items=items))
    return out

def summary_key(label, spec):
    for s in (spec or {}).get("summary_rows") or []:
        if any_term(norm_text(label), s["labels"], 0.85):
            return s["key"]
    return slug(label)

def matrix_tables(res, spec):
    cols = res["cols"]
    name_key = row_name_key(cols, spec)
    value_cols = [c for c in cols if c["role"] not in ("label", "ignore")]
    label_cols = [c for c in cols if c["role"] == "label"]
    fixed = ((spec or {}).get("layout") or {}).get("fixed_rows") or []
    tables = []
    for bi, block in enumerate(assemble_matrix(res["rows"], cols, spec)):
        rows = []
        for row_type, section, r in block["items"]:
            a = r["answer"]
            labels = {"sr_no": to_int(a.get("sr_no")) if row_type == "data" else None, "label": a.get("label")}
            for c in label_cols:
                if c["key"] not in ("sr_no", name_key):
                    labels[c["key"]] = a.get(c["key"])
            if fixed and row_type == "data":
                best = max(fixed, key=lambda f: difflib.SequenceMatcher(None, norm_text(f), norm_text(a.get("label"))).ratio())
                if difflib.SequenceMatcher(None, norm_text(best), norm_text(a.get("label"))).ratio() >= 0.6:
                    labels["label_printed"] = best
            cells = {} if row_type == "heading" else {
                c["key"]: make_cell(a.get(c["key"]), c, r["conf"].get(r["prefix"] + (c["key"],))) for c in value_cols}
            rows.append(dict(row_idx=len(rows) + 1, row_type=row_type, section=section, labels=labels, cells=cells,
                             summary_key=summary_key(a.get("label"), spec) if row_type == "summary" else None,
                             bbox=r["bbox"], crop=r["crop"], _k=r["k"]))
        tables.append(dict(table_no=bi + 1, kind="matrix", block=block["name"],
                           columns=[column_meta(c) for c in cols if c["role"] != "ignore"], rows=rows))
    return tables

def check_matrix_tables(tables, spec, checks, issues):
    """Row sums (parts -> TOTAL, dispositions -> grand total), section totals, summary rules, serial numbers.
    Returns the row keys (_k) whose row sum failed, for the re-read pass."""
    lay = (spec or {}).get("layout") or {}
    failed_rows = set()
    for t in tables:
        roles = {c["key"]: c["role"] for c in t["columns"]}
        parts = [k for k, r in roles.items() if r == "part"]
        disp = [k for k, r in roles.items() if r == "disposition"]
        has_total = "total" in roles.values()
        grand = next((k for k, r in roles.items() if r == "grand_total"), "total" if has_total else None)
        total_key = next((k for k, r in roles.items() if r == "total"), None)
        tag = f"{t['block']} " if t["block"] else ""
        def row_sums(r, scope):
            c = r["cells"]
            if total_key and parts and num(c.get(total_key)) is not None and any(num(c.get(k)) is not None for k in parts) \
                    and not any(c.get(k, {}).get("status") == "no_count" for k in parts):
                s = sum(num(c.get(k)) or 0 for k in parts)
                ok = s == num(c[total_key])
                checks.append(dict(rule="row_sum", scope=scope, expression=f"{total_key} == sum(parts)",
                                   expected=num(c[total_key]), actual=s, result="pass" if ok else "fail"))
                mark([c.get(k) for k in parts if num(c.get(k)) is not None] + [c[total_key]], "row_sum",
                     "pass" if ok else "fail")
                if not ok:
                    failed_rows.add(r["_k"])
            if disp and grand and num(c.get(grand)) is not None and any(num(c.get(k)) is not None for k in disp):
                s = sum(num(c.get(k)) or 0 for k in disp)
                ok = s == num(c[grand])
                checks.append(dict(rule="disposition_sum", scope=scope, expression=f"{grand} == sum(dispositions)",
                                   expected=num(c[grand]), actual=s, result="pass" if ok else "fail"))
                mark([c.get(k) for k in disp if num(c.get(k)) is not None] + [c[grand]], "disposition_sum",
                     "pass" if ok else "fail")
                if not ok:
                    failed_rows.add(r["_k"])
        sections = {}
        for r in t["rows"]:
            scope = f"{tag}row {r['row_idx']} {r['labels'].get('label') or ''}".strip()
            if r["row_type"] in ("data", "section_total") or (
                    r["row_type"] == "summary" and lay.get("row_sum_on_summary") and not is_pct_label(norm_text(r["labels"].get("label")))):
                row_sums(r, scope)
            if r["row_type"] == "data":
                sections.setdefault(r["section"], []).append(r)
            if r["row_type"] == "section_total":
                rows_ = sections.get(r["section"], [])
                for k in parts + disp + ([total_key] if total_key else []):
                    if num(r["cells"].get(k)) is not None and any(num(x["cells"].get(k)) is not None for x in rows_):
                        s = sum(num(x["cells"].get(k)) or 0 for x in rows_)
                        ok = s == num(r["cells"][k])
                        checks.append(dict(rule="section_total", scope=f"{tag}{r['section']} · {k}",
                                           expression=f"{k} total == sum of rows", expected=num(r["cells"][k]),
                                           actual=s, result="pass" if ok else "fail"))
                        mark([r["cells"][k]] + [x["cells"].get(k) for x in rows_ if num(x["cells"].get(k)) is not None],
                             "section_total", "pass" if ok else "fail")
        summ = {r["summary_key"]: r for r in t["rows"] if r["row_type"] == "summary"}
        for expr in (spec or {}).get("summary_rules") or []:
            if not all(n in summ for n in rule_names(expr)):
                continue
            for k in [c["key"] for c in t["columns"] if c["role"] in ("part", "total", "value")]:
                run_rule(checks, expr, f"{tag}summary · {k}", {n: summ[n]["cells"].get(k) for n in rule_names(expr)},
                         rule_id="summary_rule")
        if lay.get("row_id", "sr_no") == "sr_no":
            seq = [r["labels"]["sr_no"] for r in t["rows"] if r["row_type"] == "data" and isinstance(r["labels"]["sr_no"], int)]
            for a, b in zip(seq, seq[1:]):
                if b != a + 1 and b != 1:
                    issues.append(f"{tag}SR.NO {a} followed by {b} (row read twice, skipped or misread?)")
    return failed_rows

## 10. Layout kind `block_grid` (BS6 straight-pass monitoring: weld1, weld2, monthly handwritten)
Landscape grid: parts (one block of `params` rows each) × days (`shifts` columns each). One model call per part ×
day that has handwriting; the crop is `[part labels + parameter names | that day's columns]` with the day header on
top. Output table: one row per part × parameter, one column per day × shift.

In [ ]:
DAYS_PROMPT = """This strip is the printed header row of a monitoring sheet with day columns.
List the day numbers printed in the top row, left to right.
Return only JSON: {"days": ["21", "22", ...]}"""

def block_prompt(spec):
    lay = spec["layout"]
    params = "\n".join(f'   "{p["key"]}": number or null   (row "{p["label"]}")' for p in lay["params"])
    shifts = ", ".join(f'"{s}": SHIFT' for s in lay["shifts"])
    return f"""This crop is from the handwritten "{spec['name']}". Left: printed part details (S.NO, PART NAME, PART NO,
MODEL) and the {len(lay['params'])} parameter rows. Right: ONE day with the columns {', '.join(lay['shifts'])} (the day
number is printed on top).
Return only JSON:
{{"sno": integer or null, "part_name": string or null, "part_no": string or null, "model": string or null,
 "day": printed day number as string or null,
 "whole_day_no_plan": true if "NO PLAN" (or "NP") is written across all columns of this day, else false,
 {shifts}}}
where SHIFT is null when that column is empty, "NO PLAN" when NO PLAN / NP is written down that column only, or
{{
{params}
}}
Rules: copy exactly what is written, do not calculate. Percent values: number without %. Empty cell -> null.
A struck-out value is ignored."""

def snap_to(t, lines, tol):
    near = [l for l in lines if abs(l - t) <= tol]
    return int(min(near, key=lambda l: abs(l - t))) if near else int(round(t))

def extract_block_grid(ctx, img, raw_gray, spec):
    """The sheet is printed on a uniform grid: part blocks and day groups are found by splitting the data area evenly
    and snapping to the nearest ruled line."""
    lay = spec["layout"]
    warnings = []
    W = img.shape[1]
    gray, bw, h, v, rows, cands = basic_grid(img)
    cols, col_report = filter_columns(raw_gray, rows, cands)
    pitches = list(np.diff(rows))
    run = uniform_run(pitches, tol=0.2, min_len=8)
    if run is None or len(cols) < 4:
        raise RuntimeError(f"block grid not found (rows={len(rows)}, cols={len(cols)})")
    s = run[0]
    med = float(np.median(pitches[run[0]:run[1] + 1]))
    j = s
    while j + 1 < len(rows) and rows[j + 1] - rows[j] <= 1.6 * med:
        j += 1
    top_y, bottom_y = rows[s], rows[j]
    n_par = len(lay["params"])
    n_parts = max(1, int(round((bottom_y - top_y) / (n_par * med))))
    step = (bottom_y - top_y) / n_parts
    bounds = [snap_to(top_y + k * step, rows, 0.4 * med) for k in range(n_parts + 1)]
    blocks = list(zip(bounds[:-1], bounds[1:]))
    left = [(cols[i + 1] - cols[i], i) for i in range(len(cols) - 1) if cols[i + 1] < 0.45 * W]
    label_edge = cols[max(left)[1] + 1] if left else int(0.3 * W)
    right = cols[-1]
    hdr_top = rows[max(s - lay.get("header_bands", 2), 0)]
    ink = ink_mask(gray, h, v)

    parsed, _ = call_model(ctx, "days", with_margin(img[hdr_top:top_y, label_edge:right]), DAYS_PROMPT, 256)
    days_read = [str(d) for d in ((parsed or {}).get("days") or [])] if isinstance(parsed, dict) else []
    n_days = len(days_read) if 4 <= len(days_read) <= 16 else lay.get("days_per_sheet", 10)
    if days_read and len(days_read) != n_days:
        warnings.append(f"read {len(days_read)} day numbers - using {n_days} day groups")
    day_w = (right - label_edge) / n_days
    n_sh = len(lay["shifts"])
    gx = [snap_to(label_edge + g * day_w, cols, 0.3 * day_w / n_sh) for g in range(n_days + 1)]
    groups = list(zip(gx[:-1], gx[1:]))
    day_names = [days_read[g] if g < len(days_read) else str(g + 1) for g in range(n_days)]

    ov = img.copy()
    for y in rows:
        cv2.line(ov, (0, y), (W, y), (0, 0, 255), 1)
    for x in cols:
        cv2.line(ov, (x, 0), (x, ov.shape[0]), (255, 0, 0), 1)
    cv2.rectangle(ov, (cols[0], hdr_top), (label_edge, top_y), (255, 0, 255), 5)
    for k, (y0, y1) in enumerate(blocks):
        cv2.rectangle(ov, (cols[0] + 4, y0 + 4), (right - 4, y1 - 4), (0, 170, 0), 4)
        cv2.putText(ov, f"part {k + 1}", (cols[0] + 10, y0 + 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 170, 0), 3)
    for g, (x0, x1) in enumerate(groups):
        cv2.rectangle(ov, (x0 + 3, hdr_top), (x1 - 3, bottom_y), (0, 120, 255), 3)
        cv2.putText(ov, f"d{day_names[g]}", (x0 + 8, hdr_top - 10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 120, 255), 3)
    ctx.overlay = ctx.dbg.save("block_grid", ov)
    if SHOW_INLINE:
        show(ov, f"{ctx.pdf_rel} p{ctx.page_no} – block grid: {len(blocks)} parts green · {len(groups)} day groups orange",
             width=14)

    prompt = block_prompt(spec)
    parts = []
    for bi, (y0, y1) in enumerate(blocks):
        part = dict(sno=None, part_name=None, part_no=None, model=None, days={}, bbox=[cols[0], y0, right, y1])
        for g, (gx0, gx1) in enumerate(groups):
            amount = int(ink[y0 + 4:y1 - 4, gx0 + 4:gx1 - 4].sum() / 255)
            if amount < ST_MIN_DAY_INK:
                continue
            yb = min(y1 + BOTTOM_PAD, img.shape[0])
            label = np.vstack([img[hdr_top:top_y, cols[0]:label_edge], img[y0:yb, cols[0]:label_edge]])
            day = np.vstack([img[hdr_top:top_y, gx0:gx1], img[y0:yb, gx0:gx1]])
            crop = with_margin(np.hstack([label, np.full((label.shape[0], 12, 3), 255, np.uint8), day]))
            parsed, call = call_model(ctx, f"part{bi + 1}_day{day_names[g]}", crop, prompt, 1024)
            d = parsed if isinstance(parsed, dict) else {}
            for k in ("sno", "part_name", "part_no", "model"):
                if part[k] is None and d.get(k) not in (None, ""):
                    part[k] = to_int(d[k]) if k == "sno" else str(d[k]).strip()
            if call["error"] and not DRY_RUN:
                warnings.append(f"part {bi + 1} day {day_names[g]}: {call['error']}")
            part["days"][day_names[g]] = dict(answer=d, conf=call["conf"], crop=call["image"], bbox=[gx0, y0, gx1, y1])
        parts.append(part)
    return dict(parts=parts, days=day_names), warnings

def block_grid_tables(res, spec):
    lay = spec["layout"]
    shifts, params = lay["shifts"], lay["params"]
    columns, keymap = [], {}
    for d in res["days"]:
        for sh in shifts:
            key = f"d{slug(d)}_{slug(sh)}"
            keymap[(d, sh)] = key
            columns.append(dict(key=key, label=f"{d} {sh}", group=f"DAY {d}", role="value", type="number",
                                cell_format="plain"))
    num_col = dict(type="number")
    rows = []
    for p in res["parts"]:
        for prm in params:
            cells = {}
            for d in res["days"]:
                day = p["days"].get(d)
                for sh in shifts:
                    key = keymap[(d, sh)]
                    if day is None:
                        cells[key] = make_cell(None, num_col)
                        continue
                    a = day["answer"]
                    v = a.get(sh)
                    if a.get("whole_day_no_plan") or (isinstance(v, str) and "PLAN" in v.upper() or v == "NP"):
                        c = make_cell(None, num_col)
                        c.update(value="NO PLAN", raw="NO PLAN", source="no_plan")
                        cells[key] = c
                    elif isinstance(v, dict):
                        cells[key] = make_cell(v.get(prm["key"]), num_col, day["conf"].get((sh, prm["key"])))
                    else:
                        cells[key] = make_cell(None, num_col)
            rows.append(dict(row_idx=len(rows) + 1, row_type="data", section=p["part_name"],
                             labels=dict(sno=p["sno"], part_name=p["part_name"], part_no=p["part_no"],
                                         model=p["model"], parameter=prm["label"], param_key=prm["key"]),
                             cells=cells, bbox=p["bbox"],
                             crop=next((x["crop"] for x in p["days"].values()), None)))
    return [dict(table_no=1, kind="block_grid", block=None, columns=columns, rows=rows)]

def check_block_grid(tables, spec, checks):
    """Per part × day × shift: the spec rules over the parameter rows of that part."""
    for t in tables:
        by_part = {}
        for r in t["rows"]:
            by_part.setdefault((r["labels"]["sno"], r["section"], r["bbox"][1]), {})[r["labels"]["param_key"]] = r
        for (sno, name, _), prow in by_part.items():
            for c in t["columns"]:
                cells = {pk: r["cells"][c["key"]] for pk, r in prow.items()}
                for expr in spec.get("rules") or []:
                    run_rule(checks, expr, f"part {sno or name} · {c['label']}", cells)

## 11. Unknown forms, repairs and the page driver
**Unknown form** (`new_form` / `ambiguous`): the skeleton read decides the layout kind (printed row names →
`matrix`, handwritten entries → `record`), keys come from the printed labels, and a draft spec is written to
`forms/_drafts/` for a person to review.

**Repairs** (only rows whose totals do not add up):
1. matrix rows are re-read with a tight crop; when the new reading adds up, the changed cells are taken over with
   status `repaired` (old value kept in `previous`); when it does not, cells where the two readings disagree get
   `agreement: 0` (capped at 0.40);
2. a single cell is swapped for one of the model's alternatives (e.g. ① read as 1, alternative 7) when that makes the
   total add up → status `repaired`.

In [ ]:
INTRINSIC_CHECKS = ("pattern", "date_format")

def all_cells(tables):
    for t in tables:
        for r in t["rows"]:
            yield from r["cells"].values()

def reset_checks(tables):
    for c in all_cells(tables):
        c["checks"] = {k: v for k, v in c["checks"].items() if k in INTRINSIC_CHECKS}

def table_roles(t):
    roles = {c["key"]: c["role"] for c in t["columns"]}
    parts = [k for k, r in roles.items() if r == "part"]
    total_key = next((k for k, r in roles.items() if r == "total"), None)
    return parts, total_key

def row_adds_up(cells, parts, total_key):
    if not total_key or num(cells.get(total_key)) is None or not any(num(cells.get(k)) is not None for k in parts):
        return None
    return sum(num(cells.get(k)) or 0 for k in parts) == num(cells[total_key])

def run_checks(tables, spec, kind, issues):
    checks = []
    reset_checks(tables)
    if kind == "record":
        for t in tables:
            check_record_table(t, spec, checks)
        return checks, set()
    if kind == "block_grid":
        check_block_grid(tables, spec, checks)
        return checks, set()
    failed = check_matrix_tables(tables, spec, checks, issues)
    return checks, failed

def reread_failed_rows(tables, res, failed, warnings):
    """Matrix: rows whose totals do not add up are read again with a tight crop."""
    ks = sorted(failed)[:MAX_REREAD_ROWS]
    if not ks or DRY_RUN:
        return 0
    if len(failed) > MAX_REREAD_ROWS:
        warnings.append(f"{len(failed)} rows do not add up - only the first {MAX_REREAD_ROWS} were re-read")
    again = res["reread"](ks)
    col_by_key = {c["key"]: c for c in res["cols"]}
    changed = 0
    for t in tables:
        parts, total_key = table_roles(t)
        for r in t["rows"]:
            a = again.get(r.get("_k"))
            if not a or not a["answer"]:
                continue
            new = {k: make_cell(a["answer"].get(k), col_by_key.get(k, {}), a["conf"].get(a["prefix"] + (k,)))
                   for k in r["cells"]}
            if row_adds_up(new, parts, total_key):
                for k, c in r["cells"].items():
                    if new[k]["value"] != c["value"]:
                        new[k].update(status="repaired", previous=c["value"], repaired_by="re-read")
                        r["cells"][k] = new[k]; changed += 1
                    else:
                        c["confidence"]["agreement"] = 1
                r["crop_reread"] = a["crop"]
            else:
                for k, c in r["cells"].items():
                    if c["status"] in ("empty", "no_count") and new[k]["status"] in ("empty", "no_count"):
                        continue
                    c["confidence"]["agreement"] = int(new[k]["value"] == c["value"])
    return changed

def repair_with_alternatives(tables):
    """One cell of a row that does not add up is swapped for an alternative the model also considered, when that
    makes the row add up. The most likely such alternative wins."""
    repaired = 0
    for t in tables:
        parts, total_key = table_roles(t)
        if not parts or not total_key:
            continue
        for r in t["rows"]:
            c = r["cells"]
            if row_adds_up(c, parts, total_key) is not False:
                continue
            best = None
            for k in parts + [total_key]:
                for alt in c.get(k, {}).get("confidence", {}).get("alternatives") or []:
                    if not isinstance(alt["value"], (int, float)):
                        continue
                    trial = {kk: dict(vv) for kk, vv in c.items()}
                    trial[k]["value"] = alt["value"]
                    if row_adds_up(trial, parts, total_key) and (best is None or alt["p"] > best[2]):
                        best = (k, alt["value"], alt["p"])
            if best:
                k, v, p = best
                c[k].update(previous=c[k]["value"], value=v, status="repaired", repaired_by=f"alternative p={p}")
                repaired += 1
    return repaired

# ---- unknown forms -------------------------------------------------------------------------------------
def guess_type(label):
    n = norm_text(label)
    if re.search(r"\b(QTY|QUANTITY|NOS|COUNT|NO OF)\b", n) or n in ("S NO", "SNO", "SR NO", "S NO."):
        return "int"
    if "DATE" in n:
        return "date"
    return "text"

def generic_spec(header, skeleton, pdf_rel, kind):
    title_lines = [str(t) for t in (header or {}).get("title_lines") or []]
    title = next((t for t in title_lines if "MEENAKSHI" not in t.upper()), title_lines[0] if title_lines else Path(pdf_rel).stem)
    labels = [str(x) for x in (skeleton or {}).get("column_labels") or [] if str(x).strip()]
    fid = slug(title)[:48] or slug(Path(pdf_rel).stem)
    spec = dict(form_id=fid, version=0, name=title, generic=True,
                match=dict(title_any=[title], columns_any=labels,
                           doc_no_any=[(header or {}).get("doc_no")] if (header or {}).get("doc_no") else []),
                header={k: [k.upper().replace("_", " ")] for k in HEADER_KEYS if (header or {}).get(k)},
                layout=dict(kind=kind), columns=[], rules=[])
    if kind == "record":
        keys = set()
        for l in labels:
            k, n = slug(l), 2
            while k in keys:
                k = f"{slug(l)}_{n}"; n += 1
            keys.add(k)
            spec["columns"].append(dict(key=k, labels=[l], type=guess_type(l)))
        spec["layout"].update(footer_rows=1)
    else:
        has_sr = any(any_term(norm_text(l), ["SR NO", "S NO", "SNO"]) for l in labels[:3])
        spec["layout"].update(row_id="sr_no" if has_sr else "label", require_total=False, min_rows=8, allow_gaps=True,
                              header_terms=NAME_COL_TERMS + [l for l in labels[:2]])
    return spec

def write_draft(spec, tables, pdf_rel):
    if not WRITE_DRAFTS:
        return None
    d = FORMS_DIR / "_drafts"
    d.mkdir(parents=True, exist_ok=True)
    path = d / f"{spec['form_id']}.json"
    if path.exists():
        return path
    draft = {k: v for k, v in spec.items() if k != "generic"}
    if tables and spec["layout"]["kind"] == "matrix":
        draft["columns"] = [dict(key=c["key"], labels=[c["label"]], type=c.get("type", "number"), role=c["role"],
                                 **({"group": c["group"]} if c.get("group") else {})) for c in tables[0]["columns"]]
        draft["layout"].pop("header_terms", None)
        rows = [r["labels"].get("label") for r in tables[0]["rows"] if r["labels"].get("label")]
        if rows:
            draft["layout"]["fixed_rows_seen"] = rows
    draft["version"] = 1
    draft["_draft"] = (f"written from {pdf_rel}: check keys, types, roles and add rules, "
                       "then move this file to the forms folder")
    path.write_text(json.dumps(draft, indent=2, ensure_ascii=False), encoding="utf-8")
    return path

# ---- header output -------------------------------------------------------------------------------------
def build_header(spec, header, hconf):
    h = header if isinstance(header, dict) else {}
    others = [o for o in h.get("other_fields") or [] if isinstance(o, dict)]
    want = dict((spec or {}).get("header") or {})
    if not want or (spec or {}).get("generic"):
        want.update({"company": ["COMPANY"], "title": ["TITLE"]})
        want.update({k: [k.upper()] for k in HEADER_KEYS if h.get(k)})
        want.update({slug(o.get("label")): [o.get("label")] for o in others if o.get("label")})
    titles = [str(t) for t in h.get("title_lines") or []]
    out = {}
    for key, labels in want.items():
        raw, path = None, None
        if key == "title" and titles:
            phrases = ((spec or {}).get("match") or {}).get("title_any") or []
            i = max(range(len(titles)), key=lambda i: max([phrase_score(titles[i], p) for p in phrases] or [0])
                    - (0.5 if "MEENAKSHI" in titles[i].upper() else 0))
            raw, path = titles[i], ("title_lines", i)
        elif key in HEADER_KEYS and h.get(key) not in (None, ""):
            raw, path = h.get(key), (key,)
        else:
            for i, o in enumerate(others):
                if any(label_match(o.get("label"), l) for l in labels):
                    raw, path = o.get("value"), ("other_fields", i, "value")
                    break
        if key == "company" and raw is None:
            i = next((i for i, t in enumerate(titles) if "MEENAKSHI" in t.upper()), None)
            if i is not None:
                raw, path = titles[i], ("title_lines", i)
        cell = make_cell(raw, dict(type="date" if key in ("date", "rev_date", "org_date") else "text"),
                         hconf.get(path) if path else None)
        if key in ("rev_date", "org_date"):
            cell["checks"].pop("date_format", None)          # printed dates come in many formats
        out[key] = finalize_cell(cell)
    return out

# ---- one page ------------------------------------------------------------------------------------------
PIPELINE_VERSION = "3.0.0"
RUN_ID = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()) + f"-{os.getpid() % 10000:04d}"

def strip_internal(tables):
    for t in tables:
        for r in t["rows"]:
            r.pop("_k", None)
    return tables

def summarise(tables, header, match_status, issues):
    counts = dict(cells=0, ok=0, review=0, low=0, repaired=0, no_count=0, empty=0)
    for c in all_cells(tables):
        counts["cells"] += 1
        counts[c["status"]] = counts.get(c["status"], 0) + 1
    hdr_review = sum(c["status"] in ("review", "low") for c in header.values())
    counts["header_review"] = hdr_review
    counts["needs_review"] = bool(counts["review"] or counts["low"] or counts["repaired"] or hdr_review
                                  or issues or match_status not in ("matched", "override"))
    return counts

def process_page(pdf_rel, page, page_no, pages_total, page_dir, sha):
    ctx = PageCtx(pdf_rel, page_no, page_dir)
    t0 = time.time()
    meta = dict(debug_dir=str(page_dir))
    result = dict(schema_version="2.0",
                  document=dict(file=pdf_rel, sha256=sha, page=page_no, pages_total=pages_total),
                  run=dict(run_id=RUN_ID, model_id=MODEL_ID, pipeline_version=PIPELINE_VERSION),
                  form=None, header={}, tables=[], checks=[], issues=[], summary=None, status=None)
    try:
        text_layer = page.get_text() or ""
        img, raw_gray, angle, info = prepare_page(page, ctx.dbg)
        meta.update(deskew_deg=round(angle, 2), render=info, size_wh=(img.shape[1], img.shape[0]),
                    text_layer_chars=len(text_layer.strip()))
        orient = "portrait" if img.shape[0] >= img.shape[1] else "landscape"
        header, hcall = call_model(ctx, "header", with_margin(img[:int(img.shape[0] * HEADER_BAND_FRAC[orient])]),
                                   HEADER_PROMPT, 900)
        skeleton = None
        if not DRY_RUN:
            small = cv2.resize(img, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
            skeleton, _ = call_model(ctx, "skeleton", small, SKELETON_PROMPT, 700)
        spec, match = decide_form(pdf_rel, header, skeleton, text_layer)
        if spec is None:
            rows_printed = (skeleton or {}).get("rows_printed") if isinstance(skeleton, dict) else None
            kind = "matrix" if rows_printed else "record"
            spec = generic_spec(header, skeleton, pdf_rel, kind)
            if kind == "record" and not spec["columns"]:
                raise RuntimeError("form not recognised and no column labels read - set FORM_OVERRIDE for this file")
        kind = spec["layout"]["kind"]
        result["form"] = dict(form_id=None if spec.get("generic") else spec["form_id"],
                              form_version=None if spec.get("generic") else spec.get("version"),
                              name=spec["name"], kind=kind, match=match)
        if VERBOSE:
            print(f"    form: {result['form']['form_id'] or '(generic) ' + spec['form_id']} [{kind}] · {match['status']} "
                  f"by {match['decided_by']} · score {match['score']} · runner-up {match['runner_up']}"
                  + "".join(f"\n    ⚠ {w}" for w in match["warnings"]))
        issues, warnings = list(match["warnings"]), []
        if kind == "record":
            entries, w = extract_record(ctx, img, raw_gray, spec); warnings += w
            tables = record_table(spec, entries)
            res = None
        elif kind == "matrix":
            res, w = extract_matrix(ctx, img, spec); warnings += w
            tables = matrix_tables(res, spec)
        else:
            res, w = extract_block_grid(ctx, img, raw_gray, spec); warnings += w
            tables = block_grid_tables(res, spec)
        checks, failed = run_checks(tables, spec, kind, [])
        rereads = alt_repairs = 0
        if kind == "matrix" and failed:
            rereads = reread_failed_rows(tables, res, failed, warnings)
        if kind == "matrix":
            checks, _ = run_checks(tables, spec, kind, [])
            alt_repairs = repair_with_alternatives(tables)
        page_issues = []
        checks, _ = run_checks(tables, spec, kind, page_issues)
        issues += page_issues
        for c in all_cells(tables):
            finalize_cell(c)
        result["header"] = build_header(spec, header, hcall["conf"])
        if spec.get("generic"):
            draft = write_draft(spec, tables, pdf_rel)
            if draft:
                meta["draft_spec"] = str(draft)
                issues.append(f"unknown form - draft spec written to {draft.name}; review it and move it to the forms folder")
        result.update(tables=strip_internal(tables), checks=checks, issues=issues)
        result["summary"] = summarise(tables, result["header"], match["status"], issues)
        if DRY_RUN:
            result["status"] = "not_run"
        elif spec.get("generic"):
            result["status"] = match["status"]
        else:
            result["status"] = "review" if result["summary"]["needs_review"] else "ok"
        meta.update(warnings=warnings, rows_changed_by_reread=rereads, cells_changed_by_alternatives=alt_repairs,
                    **ctx.extra)
    except Exception as e:
        result.update(status="failed", issues=result["issues"] + [f"{type(e).__name__}: {e}"])
        meta["traceback"] = traceback.format_exc(limit=-4)
        print("    ✗", result["issues"][-1])
    finally:
        meta.update(seconds=round(time.time() - t0, 1), model_calls=len(ctx.calls),
                    overlay=str(ctx.overlay) if ctx.overlay else None)
        result["_meta"] = meta
    return result

def out_stem(pdf_rel):
    return re.sub(r"[^A-Za-z0-9/]+", "_", str(Path(pdf_rel).with_suffix(""))).strip("_")

def strip_meta(obj):
    if isinstance(obj, dict):
        return {k: strip_meta(v) for k, v in obj.items() if k != "_meta"}
    if isinstance(obj, list):
        return [strip_meta(x) for x in obj]
    return obj

def ocr_pdf(pdf_path, pdf_rel=None):
    """Single entry point: one page JSON per page -> json/<pdf>.json (+ form-shaped CSVs)."""
    import hashlib
    pdf_path = Path(pdf_path)
    pdf_rel = pdf_rel or pdf_path.name
    sha = hashlib.sha256(pdf_path.read_bytes()).hexdigest()
    doc = pymupdf.open(pdf_path)
    n = len(doc) if MAX_PAGES_PER_PDF is None else min(len(doc), MAX_PAGES_PER_PDF)
    print(f"▶ {pdf_rel}  ({len(doc)} page{'s' if len(doc) > 1 else ''})")
    pages = []
    for pno in range(n):
        page_dir = OUTPUT_DIR / "debug" / out_stem(pdf_rel) / f"p{pno + 1}"
        if page_dir.exists():
            shutil.rmtree(page_dir)
        res = process_page(pdf_rel, doc[pno], pno + 1, len(doc), page_dir, sha)
        s, m = res.get("summary") or {}, res["_meta"]
        print(f"  p{pno + 1}: {(res.get('form') or {}).get('form_id') or '—'} · {res['status']} · "
              f"cells {s.get('cells', 0)} (ok {s.get('ok', 0)}, review {s.get('review', 0)}, low {s.get('low', 0)}, "
              f"repaired {s.get('repaired', 0)}, no count {s.get('no_count', 0)}) · {m.get('model_calls')} calls · "
              f"{m.get('seconds')} s")
        for i in res.get("issues", [])[:6]:
            print("      ✗", i)
        pages.append(res)
    out = OUTPUT_DIR / "json" / f"{out_stem(pdf_rel)}.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(pages if INCLUDE_META_IN_JSON else strip_meta(pages), indent=1, ensure_ascii=False,
                              default=str), encoding="utf-8")
    for path in write_form_csvs(pages, pdf_rel):
        print("      csv:", path.relative_to(OUTPUT_DIR))
    return pages

## 12. Tables laid out like the form (CSV)

In [ ]:
STATUS_COLOURS = {"ok": "C8E6C9", "review": "FFE0B2", "low": "FFCDD2", "repaired": "E1BEE7", "no_count": "E0E0E0"}

def label_columns(t):
    if t["kind"] == "record":
        return []
    if t["kind"] == "block_grid":
        return ["sno", "part_name", "part_no", "model", "parameter"]
    keys = []
    for r in t["rows"]:
        for k in r["labels"]:
            if k not in keys and k != "param_key":
                keys.append(k)
    return keys

def table_frame(page, t):
    """A table as a DataFrame of values, printed column labels as headers, rows in sheet order."""
    lab = label_columns(t)
    head = ["BLOCK"] if t.get("block") else []
    head += ["ROW TYPE", "SECTION"] if t["kind"] == "matrix" else []
    names = head + [k.upper().replace("_", " ") for k in lab] + [
        (f"{c['group']} - {c['label']}" if c.get("group") and t["kind"] == "matrix" else c["label"]) for c in t["columns"]
        if c["role"] != "label"]
    data = []
    for r in t["rows"]:
        line = ([t["block"]] if t.get("block") else []) + ([r["row_type"], r["section"]] if t["kind"] == "matrix" else [])
        line += [r["labels"].get(k) for k in lab]
        line += [(r["cells"].get(c["key"]) or {}).get("value") for c in t["columns"] if c["role"] != "label"]
        data.append(line)
    df = pd.DataFrame(data, columns=range(len(names)), dtype=object)
    df.columns = names                                      # printed labels may repeat (two TOTAL columns)
    return df

def write_form_csvs(pages, pdf_rel):
    written = []
    for p in pages:
        for t in p.get("tables") or []:
            stem = out_stem(pdf_rel) + (f"_p{p['document']['page']}" if len(pages) > 1 else "") + \
                   (f"_{slug(t['block'])}" if t.get("block") else "")
            path = OUTPUT_DIR / "csv" / f"{stem}.csv"
            path.parent.mkdir(parents=True, exist_ok=True)
            table_frame(p, t).to_csv(path, index=False, encoding="utf-8-sig")
            written.append(path)
    return written

## 13. Run on all PDFs

In [ ]:
RESULTS = {}
t_all = time.time()
for pdf, rel in PDFS:
    RESULTS[rel] = ocr_pdf(pdf, rel)
print(f"\ndone: {len(RESULTS)} PDF(s) in {time.time() - t_all:.0f} s -> {OUTPUT_DIR / 'json'}")

def page_summary_rows(results):
    out = []
    for rel, pages in results.items():
        for p in pages:
            f, s, m = p.get("form") or {}, p.get("summary") or {}, p["_meta"]
            mt = f.get("match") or {}
            out.append(dict(pdf=rel, page=p["document"]["page"], form_id=f.get("form_id"), form=f.get("name"),
                            kind=f.get("kind"), status=p["status"], match=mt.get("status"), score=mt.get("score"),
                            decided_by=mt.get("decided_by"), runner_up=(mt.get("runner_up") or {}).get("form_id"),
                            cells=s.get("cells"), ok=s.get("ok"), review=s.get("review"), low=s.get("low"),
                            repaired=s.get("repaired"), no_count=s.get("no_count"), empty=s.get("empty"),
                            checks_failed=sum(c["result"] == "fail" for c in p.get("checks") or []),
                            issues=len(p.get("issues") or []), calls=m.get("model_calls"), seconds=m.get("seconds")))
    return out

summary = pd.DataFrame(page_summary_rows(RESULTS))
summary.to_csv(OUTPUT_DIR / "pages_summary.csv", index=False, encoding="utf-8-sig")
summary

## 14. Review workbook – `review.xlsx`
- **pages**: one line per page (form, match, cell counts per status)
- **cells**: one line per cell with value, raw, confidence, alternatives, checks and status – ready for a database
- one sheet per page table, laid out like the form; every cell is coloured by status and carries a comment with the
  raw reading, the confidence, the alternatives and the checks.
  green `ok` · orange `review` · red `low` · purple `repaired` · grey `no_count` · white `empty`

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.comments import Comment
from openpyxl.utils import get_column_letter

def cell_note(c):
    conf = c.get("confidence") or {}
    parts = [f"status: {c['status']}", f"raw: {c.get('raw')!r}"]
    if conf.get("final") is not None:
        parts.append(f"confidence: {conf['final']} (model {conf.get('model')})")
    if conf.get("alternatives"):
        parts.append("alternatives: " + ", ".join(f"{a['value']} ({a['p']})" for a in conf["alternatives"]))
    if c.get("previous") is not None:
        parts.append(f"before repair: {c['previous']} ({c.get('repaired_by')})")
    if c.get("checks"):
        parts.append("checks: " + ", ".join(f"{k} {v}" for k, v in c["checks"].items()))
    if conf.get("agreement") is not None:
        parts.append(f"second read agrees: {bool(conf['agreement'])}")
    return "\n".join(parts)

def long_cells(results):
    out = []
    for rel, pages in results.items():
        for p in pages:
            base = dict(pdf=rel, page=p["document"]["page"], form_id=(p.get("form") or {}).get("form_id"))
            for key, c in (p.get("header") or {}).items():
                out.append(dict(base, table=0, block=None, row_idx=None, row_type="header", section=None,
                                row_label=None, col_key=key, col_label=key, value=c["value"], raw=c["raw"],
                                source=c["source"], conf_model=c["confidence"]["model"],
                                conf_final=c["confidence"]["final"], agreement=None, alternatives=None,
                                checks=None, previous=None, status=c["status"]))
            for t in p.get("tables") or []:
                labels = {c["key"]: c["label"] for c in t["columns"]}
                for r in t["rows"]:
                    rl = r["labels"].get("label") or r["labels"].get("parameter") or r["labels"].get("part_name")
                    for key, c in r["cells"].items():
                        out.append(dict(base, table=t["table_no"], block=t.get("block"), row_idx=r["row_idx"],
                                        row_type=r["row_type"], section=r.get("section"), row_label=rl, col_key=key,
                                        col_label=labels.get(key), value=c["value"], raw=c["raw"], source=c["source"],
                                        conf_model=c["confidence"]["model"], conf_final=c["confidence"]["final"],
                                        agreement=c["confidence"]["agreement"],
                                        alternatives=json.dumps(c["confidence"]["alternatives"], ensure_ascii=False)
                                        if c["confidence"]["alternatives"] else None,
                                        checks=json.dumps(c["checks"]) if c["checks"] else None,
                                        previous=c.get("previous"), status=c["status"]))
    return out

def write_sheet_df(ws, df):
    ws.append(list(df.columns))
    for row in df.itertuples(index=False):
        ws.append([None if (isinstance(v, float) and np.isnan(v)) else v for v in row])
    for cell in ws[1]:
        cell.font = Font(bold=True)

def build_review_xlsx(results, path):
    wb = Workbook()
    ws = wb.active; ws.title = "pages"
    write_sheet_df(ws, pd.DataFrame(page_summary_rows(results)))
    ws.freeze_panes = "A2"
    cells_df = pd.DataFrame(long_cells(results))
    wsc = wb.create_sheet("cells")
    if len(cells_df):
        write_sheet_df(wsc, cells_df)
        st_col = list(cells_df.columns).index("status") + 1
        for i in range(2, wsc.max_row + 1):
            colour = STATUS_COLOURS.get(wsc.cell(i, st_col).value)
            if colour:
                wsc.cell(i, st_col).fill = PatternFill("solid", fgColor=colour)
        wsc.freeze_panes = "A2"; wsc.auto_filter.ref = wsc.dimensions
    used = set()
    for rel, pages in results.items():
        for p in pages:
            for t in p.get("tables") or []:
                base = re.sub(r"[^A-Za-z0-9]+", "_", Path(rel).stem)[:20]
                name = f"{base}_p{p['document']['page']}" + (f"_{t['block']}" if t.get("block") else "")
                name, n = name[:31], 2
                while name in used:
                    name = f"{name[:28]}_{n}"; n += 1
                used.add(name)
                sh = wb.create_sheet(name)
                hdr = p.get("header") or {}
                sh.append([f"{rel} · page {p['document']['page']} · {(p.get('form') or {}).get('name')} · "
                           f"status {p['status']}"])
                sh.append(["; ".join(f"{k}: {c['value']}" for k, c in hdr.items() if c["value"] is not None)])
                df = table_frame(p, t)
                sh.append(list(df.columns))
                for cell in sh[3]:
                    cell.font = Font(bold=True); cell.alignment = Alignment(wrap_text=True, vertical="top")
                value_cols = [c for c in t["columns"] if c["role"] != "label"]
                first_val = len(df.columns) - len(value_cols) + 1
                for ri, (r, row) in enumerate(zip(t["rows"], df.itertuples(index=False))):
                    sh.append([None if (isinstance(v, float) and np.isnan(v)) else v for v in row])
                    excel_row = 4 + ri
                    for ci, c in enumerate(value_cols):
                        cell = r["cells"].get(c["key"])
                        if not cell:
                            continue
                        xc = sh.cell(excel_row, first_val + ci)
                        colour = STATUS_COLOURS.get(cell["status"])
                        if colour:
                            xc.fill = PatternFill("solid", fgColor=colour)
                        if cell["status"] != "empty" or cell.get("raw"):
                            xc.comment = Comment(cell_note(cell), "OCR", width=260, height=140)
                    if r["row_type"] in ("heading", "section_total", "summary"):
                        for xc in sh[excel_row][:first_val - 1]:
                            xc.font = Font(bold=True)
                sh.freeze_panes = sh.cell(4, first_val)
                for i in range(1, len(df.columns) + 1):
                    sh.column_dimensions[get_column_letter(i)].width = 30 if i < first_val else 11
    leg = wb.create_sheet("legend")
    for st, colour in [("ok", STATUS_COLOURS["ok"]), ("review", STATUS_COLOURS["review"]), ("low", STATUS_COLOURS["low"]),
                       ("repaired", STATUS_COLOURS["repaired"]), ("no_count", STATUS_COLOURS["no_count"]), ("empty", None)]:
        leg.append([st, {"ok": f"confidence >= {CONF_OK}", "review": f"{CONF_REVIEW} <= confidence < {CONF_OK}",
                         "low": f"confidence < {CONF_REVIEW}, failed check or required value missing",
                         "repaired": "changed by a re-read or an alternative so the total adds up - check it",
                         "no_count": "tally strokes without a written count", "empty": "blank or dash"}[st]])
        if colour:
            leg.cell(leg.max_row, 1).fill = PatternFill("solid", fgColor=colour)
    wb.save(path)
    return path

print("written:", build_review_xlsx(RESULTS, OUTPUT_DIR / "review.xlsx"))

## 15. Review page – `review.html`
Every page as a table laid out like the form, cells coloured by status, the reading details on hover, the row's crop
one click away. **Cells are editable**: edited cells turn blue, and *Download corrections* saves
`corrections.json` (`file, page, table, row, column, old, new`) for the database / backend.
Download `ocr_v3_output.zip` (last cell), unzip and open `review.html` so the crop images resolve.

In [ ]:
def esc(x): return html.escape("" if x is None else str(x))

def rel_link(path):
    return Path(os.path.relpath(path, OUTPUT_DIR)).as_posix() if path else ""

REVIEW_CSS = """
:root{--ok:#c8e6c9;--review:#ffe0b2;--low:#ffcdd2;--repaired:#e1bee7;--no_count:#e0e0e0;--edited:#bbdefb}
body{font-family:system-ui,Segoe UI,sans-serif;margin:16px;background:#fafafa;color:#222}
h2{margin:28px 0 6px;font-size:18px}.meta{color:#555;font-size:12px;margin:2px 0 8px}
.page{background:#fff;border:1px solid #ddd;border-radius:8px;padding:12px;margin:12px 0;overflow-x:auto}
table{border-collapse:collapse;font-size:12px;margin:6px 0}th,td{border:1px solid #ccc;padding:3px 6px;
white-space:nowrap}th{background:#f1f1f1;position:sticky;top:0}td.lab{background:#fafafa}
td.ok{background:var(--ok)}td.review{background:var(--review)}td.low{background:var(--low)}
td.repaired{background:var(--repaired)}td.no_count{background:var(--no_count)}td.edited{background:var(--edited)!important}
tr.heading td.lab,tr.summary td.lab,tr.section_total td.lab{font-weight:600}
.bar{position:sticky;top:0;background:#fafafa;padding:6px 0;z-index:5;border-bottom:1px solid #ddd}
.legend span{display:inline-block;padding:2px 8px;margin-right:4px;border-radius:4px;font-size:12px}
button{margin-right:6px}.issue{color:#c62828;font-size:12px}.hide-ok td.ok{background:#fff;color:#bbb}
"""

REVIEW_JS = """
document.addEventListener('input', e => {
  const td = e.target.closest('td[data-col]'); if (!td) return;
  td.classList.toggle('edited', td.textContent.trim() !== td.dataset.orig);
  document.getElementById('nedit').textContent = document.querySelectorAll('td.edited').length;
});
function download() {
  const out = [...document.querySelectorAll('td.edited')].map(td => ({file: td.dataset.file, page: +td.dataset.page,
    table: +td.dataset.table, row: +td.dataset.row, column: td.dataset.col, old: td.dataset.orig,
    new: td.textContent.trim()}));
  const a = document.createElement('a');
  a.href = URL.createObjectURL(new Blob([JSON.stringify(out, null, 1)], {type: 'application/json'}));
  a.download = 'corrections.json'; a.click();
}
function onlyReview(on) { document.body.classList.toggle('hide-ok', on); }
"""

def cell_td(c, file, page, table, row, col):
    v = c.get("value")
    shown = "" if v is None else str(v)
    return (f"<td class='{esc(c['status'])}' contenteditable='true' title='{esc(cell_note(c))}' "
            f"data-file='{esc(file)}' data-page='{page}' data-table='{table}' data-row='{row}' data-col='{esc(col)}' "
            f"data-orig='{esc(shown)}'>{esc(shown)}</td>")

def page_html(rel, p):
    f, s = p.get("form") or {}, p.get("summary") or {}
    mt = f.get("match") or {}
    out = [f"<div class='page'><h2>{esc(rel)} · page {p['document']['page']}</h2>",
           f"<div class='meta'><b>{esc(f.get('name'))}</b> ({esc(f.get('form_id') or 'unknown form')}, {esc(f.get('kind'))}) · "
           f"status <b>{esc(p['status'])}</b> · match {esc(mt.get('status'))} {esc(mt.get('score'))} by "
           f"{esc(mt.get('decided_by'))} · cells {s.get('cells', 0)}: ok {s.get('ok', 0)}, review {s.get('review', 0)}, "
           f"low {s.get('low', 0)}, repaired {s.get('repaired', 0)}, no count {s.get('no_count', 0)}</div>"]
    out += [f"<div class='issue'>✗ {esc(i)}</div>" for i in p.get("issues") or []][:12]
    hdr = p.get("header") or {}
    if hdr:
        out.append("<table><tr>" + "".join(f"<th>{esc(k)}</th>" for k in hdr) + "</tr><tr>"
                   + "".join(cell_td(c, rel, p["document"]["page"], 0, 0, k) for k, c in hdr.items()) + "</tr></table>")
    for t in p.get("tables") or []:
        lab = label_columns(t)
        vals = [c for c in t["columns"] if c["role"] != "label"]
        if t.get("block"):
            out.append(f"<div class='meta'><b>block {esc(t['block'])}</b></div>")
        groups = [c.get("group") for c in vals]
        out.append("<table><thead>")
        if t["kind"] != "record" and any(groups):
            out.append("<tr>" + f"<th colspan='{len(lab) + (2 if t['kind'] == 'matrix' else 0) + 1}'></th>"
                       + "".join(f"<th>{esc(g or '')}</th>" for g in groups) + "</tr>")
        out.append("<tr><th>#</th>" + ("<th>row type</th><th>section</th>" if t["kind"] == "matrix" else "")
                   + "".join(f"<th>{esc(k)}</th>" for k in lab) + "".join(f"<th>{esc(c['label'])}</th>" for c in vals)
                   + "</tr></thead><tbody>")
        for r in t["rows"]:
            crop = rel_link(r.get("crop"))
            idx = f"<a href='{esc(crop)}' target='_blank'>{r['row_idx']}</a>" if crop else str(r["row_idx"])
            line = f"<tr class='{esc(r['row_type'])}'><td class='lab'>{idx}</td>"
            if t["kind"] == "matrix":
                line += f"<td class='lab'>{esc(r['row_type'])}</td><td class='lab'>{esc(r.get('section'))}</td>"
            line += "".join(f"<td class='lab'>{esc(r['labels'].get(k))}</td>" for k in lab)
            line += "".join(cell_td(r["cells"][c["key"]], rel, p["document"]["page"], t["table_no"], r["row_idx"], c["key"])
                            if c["key"] in r["cells"] else "<td></td>" for c in vals)
            out.append(line + "</tr>")
        out.append("</tbody></table>")
    fails = [c for c in p.get("checks") or [] if c["result"] == "fail"]
    if fails:
        out.append("<details><summary>failed checks (" + str(len(fails)) + ")</summary><table><tr><th>rule</th>"
                   "<th>scope</th><th>expression</th><th>expected</th><th>actual</th></tr>"
                   + "".join(f"<tr><td>{esc(c['rule'])}</td><td>{esc(c['scope'])}</td><td>{esc(c['expression'])}</td>"
                             f"<td>{esc(c['expected'])}</td><td>{esc(c['actual'])}</td></tr>" for c in fails)
                   + "</table></details>")
    if p["_meta"].get("overlay"):
        out.append(f"<details><summary>layout overlay</summary><img style='max-width:100%' "
                   f"src='{esc(rel_link(p['_meta']['overlay']))}'></details>")
    out.append("</div>")
    return "\n".join(out)

def build_review_html(results):
    legend = "".join(f"<span style='background:var(--{k})'>{k}</span>" for k in
                     ("ok", "review", "low", "repaired", "no_count", "edited"))
    body = [f"<div class='bar'><b>OCR review</b> · {esc(MODEL_ID)} · {time.strftime('%Y-%m-%d %H:%M')} · "
            f"<span class='legend'>{legend}</span> <button onclick='download()'>Download corrections "
            f"(<span id='nedit'>0</span>)</button><label><input type='checkbox' onchange='onlyReview(this.checked)'> "
            f"fade ok cells</label></div>"]
    for rel, pages in results.items():
        body += [page_html(rel, p) for p in pages]
    return (f"<!doctype html><html><head><meta charset='utf-8'><title>OCR review</title><style>{REVIEW_CSS}</style>"
            f"</head><body>{''.join(body)}<script>{REVIEW_JS}</script></body></html>")

review_path = OUTPUT_DIR / "review.html"
review_path.write_text(build_review_html(RESULTS), encoding="utf-8")
print("review page:", review_path)

## 16. Backend export – `backend/<pdf>.json`
The FastAPI backend (`app/services/ocr_client.py`, `OcrResult`) takes one flat list of fields per document:
`fieldName`, `value`, `rawValue`, `confidence`, `status`, `scope` (`HEADER` / `LINE`), `rowIndex`, `pageIndex`.
`confidence` is the final score, so the backend's VERIFIED / NEEDS_REVIEW / ERROR thresholds apply to it. Field
names are the spec's column keys: they must match the backend template's `field_key`s.

In [ ]:
def backend_result(rel, pages):
    fields = []
    for p in pages:
        pi = p["document"]["page"] - 1
        for key, c in (p.get("header") or {}).items():
            fields.append(dict(fieldName=key, value=c["value"], rawValue=c["raw"], confidence=c["confidence"]["final"],
                               status=c["status"], scope="HEADER", rowIndex=None, pageIndex=pi))
        row_base = 0
        for t in p.get("tables") or []:
            for r in t["rows"]:
                for key, c in r["cells"].items():
                    fields.append(dict(fieldName=key, value=c["value"], rawValue=c["raw"],
                                       confidence=c["confidence"]["final"], status=c["status"], scope="LINE",
                                       rowIndex=row_base + r["row_idx"] - 1, pageIndex=pi,
                                       rowLabel=r["labels"].get("label") or r["labels"].get("parameter"),
                                       block=t.get("block"),
                                       boundingBox=dict(x=r["bbox"][0], y=r["bbox"][1], width=r["bbox"][2] - r["bbox"][0],
                                                        height=r["bbox"][3] - r["bbox"][1]) if r.get("bbox") else None))
            row_base += len(t["rows"])
    first = pages[0] if pages else {}
    return dict(reportType=(first.get("form") or {}).get("form_id"), documentId=(first.get("document") or {}).get("sha256"),
                engine=f"{MODEL_ID} / pipeline {PIPELINE_VERSION}", pageCount=len(pages), fields=fields)

for rel, pages in RESULTS.items():
    path = OUTPUT_DIR / "backend" / f"{out_stem(rel)}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(backend_result(rel, pages), indent=1, ensure_ascii=False, default=str), encoding="utf-8")
print("backend files:", OUTPUT_DIR / "backend")

## 17. Debugging: one page, and the model-call report
`inspect_page(pdf_name_part, page)` shows the overlay, every model input with its raw answer and weakest token, the
issues and the JSON. `debug.html` puts all model calls of all pages side by side.

In [ ]:
def find_result(name_part, page=1):
    for rel, pages in RESULTS.items():
        if name_part.lower() in rel.lower():
            return rel, next((p for p in pages if p["document"]["page"] == page), None)
    return None, None

def inspect_page(name_part, page=1, max_calls=8, show_json=True):
    rel, res = find_result(name_part, page)
    if res is None:
        print("not found"); return
    m = res["_meta"]
    print(f"{rel} p{page}: {(res.get('form') or {}).get('form_id')} · status {res['status']}")
    print("match:", json.dumps((res.get("form") or {}).get("match"), ensure_ascii=False))
    if m.get("overlay"):
        show(cv2.imread(m["overlay"]), "overlay", width=12)
    log = Path(m["debug_dir"]) / "model_io.jsonl"
    if log.exists():
        for line in log.read_text(encoding="utf-8").splitlines()[:max_calls]:
            c = json.loads(line)
            show(cv2.imread(c["image"]), f"{c['name']} · sent {c['image_wh']}", width=12)
            weak = sorted(c.get("conf", {}).items(), key=lambda kv: kv[1]["model"])[:5]
            print("raw:", (c.get("raw") or "")[:1200], "\nweakest values:", weak, "\nerror:", c.get("error"))
    print("issues:", *res.get("issues", []), sep="\n  ")
    print("warnings:", *m.get("warnings", []), sep="\n  ")
    if show_json:
        print(json.dumps(strip_meta(res), indent=1, ensure_ascii=False, default=str)[:6000])

def build_debug_report(results):
    css = ("body{font-family:system-ui,sans-serif;margin:24px;background:#fafafa}.page{background:#fff;border:1px solid "
           "#ddd;border-radius:8px;padding:12px;margin:14px 0}img{max-width:100%;border:1px solid #bbb}pre{background:"
           "#f3f3f3;padding:8px;white-space:pre-wrap;font-size:12px;max-height:420px;overflow:auto}.call{display:flex;"
           "gap:12px;margin:8px 0}.call>div{flex:1;min-width:0}.meta{color:#666;font-size:12px}")
    out = [f"<html><head><meta charset='utf-8'><title>OCR model calls</title><style>{css}</style></head><body>"]
    for rel, pages in results.items():
        for p in pages:
            m = p["_meta"]
            out.append(f"<div class='page'><b>{esc(rel)} p{p['document']['page']}</b> · "
                       f"{esc((p.get('form') or {}).get('form_id'))} · {esc(p['status'])}")
            log = Path(m["debug_dir"]) / "model_io.jsonl"
            if log.exists():
                for line in log.read_text(encoding="utf-8").splitlines():
                    c = json.loads(line)
                    weak = sorted(c.get("conf", {}).items(), key=lambda kv: kv[1]["model"])[:6]
                    out.append(f"<div class='call'><div><div class='meta'>{esc(c['name'])} · {esc(c['image_wh'])} · "
                               f"{esc(c.get('seconds'))} s · {esc(c.get('error') or 'ok')}</div>"
                               f"<img src='{rel_link(c['image'])}'></div><div><pre>{esc(c.get('raw'))}</pre>"
                               f"<div class='meta'>weakest: {esc(weak)}</div></div></div>")
            out.append("</div>")
    return "\n".join(out + ["</body></html>"])

(OUTPUT_DIR / "debug.html").write_text(build_debug_report(RESULTS), encoding="utf-8")
if RESULTS:
    inspect_page(next(iter(RESULTS)), 1, max_calls=3, show_json=False)

## 18. One-off: OCR a single PDF (upload or Drive path)

In [ ]:
# path = "/content/drive/MyDrive/MeenakshiPublic/some_new_scan.pdf"
# pages = ocr_pdf(path)
# print(json.dumps(strip_meta(pages), indent=1, ensure_ascii=False, default=str)[:5000])
#
# upload from your computer:
# from google.colab import files
# for name in files.upload():
#     pages = ocr_pdf(name); print(json.dumps(strip_meta(pages), indent=1, ensure_ascii=False, default=str)[:5000])

In [ ]:
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print(f"zipped: {archive} ({Path(archive).stat().st_size / 1e6:.1f} MB)")